# Evaluating the fine-tuned GENCO, and the unresolved-case warning

The protocol is written in the first cell, before any fine-tuned output is seen. Every comparison is paired (the same
cases through both models) and uses data that neither training nor model selection touched:

1. Clustered cases: the 870 frozen-test contingencies, and one soft-footprint N-20 case per frozen-test operating point
   (a construction held out of fine-tuning).
2. Forgetting: released N-2, N-10 and N-20 scenarios, with a non-inferiority margin declared in advance.
3. Near the fold: the 120 storms, with their folds and AC solutions recomputed as in `storms`. Beyond the fold the alarm
   must still hold.
4. The warning head: a logistic regression on GENCO's outputs and embeddings, trained on the fine-tuning cases, that
   predicts whether the specified solver will leave a case unresolved. Its threshold is fixed on the validation cases.
   The fitted head, its feature definitions and every case's score are written to `unresolved_warning/`.

Set `FINETUNE_SEED` in the settings cell to the run being evaluated; every test case is the same for both runs.

Runtime: T4 GPU, about 3 GB of downloads, 45 to 70 minutes. It reads `finetune_seed<FINETUNE_SEED>.zip` from `MyDrive/genco_runs`
(or from the Colab file pane). Output: `evaluate_finetuned_seed<FINETUNE_SEED>.zip`, saved to the same Drive folder.

## Settings (frozen)

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
LOCAL_DIR = "genco-pf-contingency"
FINETUNE_SEED = 0                     # which fine-tuning run to evaluate: 0 or 1
OUT_DIR = f"evaluate_finetuned_seed{FINETUNE_SEED}"
GRAPHKIT_DIR = "gridfm-graphkit"
GRAPHKIT_BRANCH = "genco-paper-repro"
NETWORK = "Texas2k_case1_2016summerpeak"
MODEL_DIR = "model/training_bs_16"
GRAPHKIT_COMMIT = "95b90eecc7f5c2c86a9dd776f0c3f40434def22f"   # pinned upstream code
HF_REVISION = "5c625b00f5b79c8a7d4d27ecc133d4ecdf573af7"       # pinned model and data revision (unchanged since 1 October 2026)
T1_ZIP = f"finetune_seed{FINETUNE_SEED}.zip"
T1_DIR = f"finetune_seed{FINETUNE_SEED}"

LEVEL_BASE = 2
N2_PARTS = [0, 1, 2, 3, 4, 5, 6, 7, 16]   # storms (0-1), the head's training cases (2-5), frozen test (6, 16), forgetting (7)
BASE_PARTS = N2_PARTS
FORGET_PARTS = {2: [7], 10: [0], 20: [0]}
N_FORGET = 200
N_CHECK = 16
HEAD_TRAIN_BASES_N = 300
HEAD_RECALL = 0.95
NONINF_MARGIN_PU = 0.001
BOOTSTRAP = 1000
INFER_BATCH = 16
LINE_MARGIN = 0.0454
POLICY = {"tol_pu": 1e-8, "max_iter": 30, "order": ["genco", "base", "dc", "flat"], "line_limit": 1.0, "voltage_buses": "pq", "low_voltage_flag": 0.5,
          "recovery": {"max_iter": 100, "homotopy_steps": [0.25, 0.5, 0.75, 1.0]}, "screen_margin": 0.0454, "enforce_q_limits": False, "q_limit_rounds": 15}
LAM_STEP, LAM_WIDTH, LAM_MIN, LAM_CAP = 0.02, 0.0025, 0.5, 2.0      # storm's nose search
REGION_HOPS, FOOT_HOPS, FOOT_SCALE, BETWEENNESS_SAMPLES = 4, 6, 2.0, 400
TOL, MAX_NR = 1e-8, 30
MAX_DOWNLOAD_GB = 5
SEED = 0
F1_CASES_B64 = "..."
K1_CASES_B64 = "..."
K1_SWEEP_B64 = "..."

In [ ]:
# Google Drive: results are copied to MyDrive/genco_runs at the end (Colab asks for permission once)
import os, shutil
DRIVE_DIR = "/content/drive/MyDrive/genco_runs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("results will be saved to", DRIVE_DIR)
except Exception as e:  # not running on Colab, or Drive declined
    DRIVE_DIR = None
    print("Google Drive not available, results stay in the session:", e)


def save_to_drive(path):
    """Copies a result file to Google Drive and starts a browser download as well."""
    if DRIVE_DIR:
        shutil.copy(path, DRIVE_DIR)
        print(f"saved {os.path.basename(path)} to {DRIVE_DIR}")
    try:
        from google.colab import files
        files.download(path)
    except Exception:
        pass


In [ ]:
# [GPU] install and import

import importlib.util, os, subprocess, sys, time


def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")


import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.\n"
          "The notebook also works on CPU, but the timing results would not be meaningful.")

if not os.path.isdir(GRAPHKIT_DIR):
    run(f"git clone -q -b {GRAPHKIT_BRANCH} https://github.com/gridfm/gridfm-graphkit.git {GRAPHKIT_DIR}")
    run(f"git -C {GRAPHKIT_DIR} checkout -q {GRAPHKIT_COMMIT}")   # pinned: the commit every run used
need = {"torch_geometric": "torch_geometric", "lightning": "lightning", "pytorch_lightning": "pytorch-lightning",
        "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow", "seaborn": "seaborn", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    run(f"{sys.executable} -m pip install -q " + " ".join(missing))

import torch_geometric   # imported before the replacement below, so PyTorch Geometric keeps its own native code paths
USED_SHIM = importlib.util.find_spec("torch_scatter") is None
if USED_SHIM:
    shim = os.path.abspath(os.path.join("shims", "torch_scatter"))
    os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "__init__.py"), "w") as f:
        f.write('"""Small pure-PyTorch stand-in for torch_scatter (only the functions gridfm-graphkit uses)."""\nimport torch\n\n__version__ = "0.0.0-shim"\n\n\ndef _prep(src, index, dim):\n    dim = dim if dim >= 0 else src.dim() + dim\n    if index.dim() != src.dim():\n        shape = [1] * src.dim()\n        shape[dim] = -1\n        index = index.view(shape).expand_as(src)\n    return index, dim\n\n\ndef _size(src, index, dim, dim_size):\n    size = list(src.shape)\n    if dim_size is None:\n        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0\n    size[dim] = dim_size\n    return size\n\n\ndef scatter_add(src, index, dim=-1, out=None, dim_size=None):\n    index, dim = _prep(src, index, dim)\n    if out is None:\n        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)\n    return out.scatter_add_(dim, index, src)\n\n\nscatter_sum = scatter_add\n\n\ndef scatter_mean(src, index, dim=-1, out=None, dim_size=None):\n    total = scatter_add(src, index, dim, None, dim_size)\n    idx, d = _prep(src, index, dim)\n    count = torch.zeros_like(total).scatter_add_(d, idx, torch.ones_like(src)).clamp_(min=1)\n    return total / count\n\n\ndef scatter_max(src, index, dim=-1, out=None, dim_size=None):\n    idx, d = _prep(src, index, dim)\n    res = torch.full(_size(src, idx, d, dim_size), float("-inf"), dtype=src.dtype, device=src.device)\n    res = res.scatter_reduce(d, idx, src, reduce="amax", include_self=True)\n    res[torch.isinf(res)] = 0\n    return res, None\n\n\ndef scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):\n    if reduce in ("sum", "add"):\n        return scatter_add(src, index, dim, out, dim_size)\n    if reduce == "mean":\n        return scatter_mean(src, index, dim, out, dim_size)\n    if reduce == "max":\n        return scatter_max(src, index, dim, out, dim_size)[0]\n    raise NotImplementedError(reduce)\n')
    sys.path.insert(0, os.path.abspath("shims"))
    print("torch_scatter is not installed: using a small built-in replacement")
sys.path.insert(0, os.path.abspath(GRAPHKIT_DIR))

import gridfm_graphkit
from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, get_task_transforms, get_task
from gridfm_graphkit.training.callbacks import adapt_state_dict_for_model
from gridfm_graphkit.datasets.hetero_preprocess import build_hetero_data_for_scenario, merge_agg_gen_into_bus
from gridfm_graphkit.datasets.globals import VM_OUT, VA_OUT
from torch_geometric.loader import DataLoader
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch_geometric", torch_geometric.__version__, "| gridfm-graphkit from", os.path.abspath(GRAPHKIT_DIR), "| device:", DEVICE)

In [ ]:
# lightsim2grid: open-source C++ Newton-Raphson (KLU sparse solver), used here as the professional AC solver
import importlib.util, subprocess, sys
if importlib.util.find_spec("lightsim2grid") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightsim2grid"], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print("lightsim2grid installed:", importlib.util.find_spec("lightsim2grid") is not None)

In [ ]:
# PYPOWER (the Python port of MATPOWER): the independent implementation used for the validation
import importlib.util, subprocess, sys
if importlib.util.find_spec("pypower") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pypower"], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError("pip install pypower failed: " + r.stderr[-800:])
import pypower
from importlib.metadata import version as _pkg_version
print("PYPOWER", _pkg_version("PYPOWER"))

In [ ]:
import base64
GENCO_VERIFY_SOURCE = base64.b64decode('...').decode()
exec(GENCO_VERIFY_SOURCE)
print('genco_verify loaded; lightsim2grid (KLU):', HAVE_KLU)

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on", "line"]:
        if key in A:
            out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ===================== GENCO helpers =====================
from scipy.sparse.csgraph import dijkstra


class ScenarioEval:
    """Truth-based checks of any voltage state for one stored scenario (from extract_scenario)."""

    def __init__(self, d):
        self.d = d
        nb, on = d["nb"], d["on"]
        self.Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
        self.Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
        self.pv, self.pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.rated = on & (d["rate"] > 0)
        Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
        self.ld_t = self.loading(Vt)
        self.over_t = self.ld_t >= 1.0
        self.vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / d["rate"][self.rated]

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def metrics(self, Vm, Va_deg):
        d = self.d
        V = self.state(Vm, Va_deg)
        S = V * np.conj(self.Y @ V) - self.Sspec
        Pm, Qm = np.abs(S.real), np.abs(S.imag)
        ld = self.loading(V)
        Vabs = np.abs(V)
        vio = d["PQ"] & ((Vabs < d["min_vm_pu"]) | (Vabs > d["max_vm_pu"]))
        err = np.abs(Vabs - d["Vm"])
        return {"err_vm_max": float(err[d["PQ"]].max()) if d["PQ"].any() else np.nan,
                "err_load_max": float(np.max(np.abs(ld - self.ld_t))) if len(ld) else np.nan,
                "mismatch_pu": float(max(Pm[np.r_[self.pv, self.pq]].max(), Qm[self.pq].max())),
                "res_p99_MW": float(np.percentile(Pm[self.nonref], 99) * SN_MVA),
                "missed_overloads": int((self.over_t & (ld < 1.0)).sum()),
                "missed_vviol": int((self.vio_t & ~vio).sum())}

    def bus_errors(self, Vm):
        return np.where(self.d["PQ"], np.abs(Vm - self.d["Vm"]), np.nan)


def thevenin_pq(d, block=200):
    """|Z_ii| for every PQ bus with PV and slack buses held at fixed voltage: how much the voltage at a bus
    moves per unit of current injected there (large = electrically weak spot)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    pq = np.nonzero(d["PQ"])[0]
    lu = splu(Y[pq][:, pq].tocsc())
    n = len(pq); z = np.empty(n, complex)
    for s in range(0, n, block):
        idx = np.arange(s, min(n, s + block))
        E = np.zeros((n, len(idx)), complex); E[idx, np.arange(len(idx))] = 1.0
        X = lu.solve(E)
        z[idx] = X[idx, np.arange(len(idx))]
    return pq, np.abs(z)


def hops_to_voltage_control(d):
    """Number of in-service lines between each bus and the nearest PV or slack bus."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    src = np.nonzero(d["PV"] | d["REF"])[0]
    dist = dijkstra(A, directed=False, unweighted=True, indices=src, min_only=True)
    return dist


def dc_solve_seconds(d, reps=3):
    """Time of one DC power flow (one sparse solve on the susceptance matrix), for the cost comparison."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    keep = np.nonzero(~d["REF"])[0]
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter()
    for _ in range(reps):
        B = (-Y.imag)[keep][:, keep].tocsc()
        splu(B).solve(P)
    return (time.perf_counter() - t0) / reps


# ===================== outage construction =====================
from scipy.sparse.csgraph import connected_components
try:
    from lightsim2grid.algorithm import NR_KLU as _KLU
    HAVE_KLU = True
except Exception:
    try:
        from lightsim2grid.solver import KLUSolver as _KLU
        HAVE_KLU = True
    except Exception:
        HAVE_KLU = False


def system_of(d):
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    return Y, S, np.nonzero(d["REF"])[0], np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]


def klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol=1e-8):
    # lightsim2grid's C++ Newton-Raphson with the KLU sparse solver. Returns V, converged, iterations, seconds.
    w = np.zeros(Y.shape[0]); w[ref] = 1.0 / len(ref)
    Yc = Y.tocsc()
    s = _KLU()
    t0 = time.perf_counter()
    ok = s.solve(Yc, np.asarray(V0, complex), np.asarray(S, complex), ref.astype(np.int32), w,
                 pv.astype(np.int32), pq.astype(np.int32), int(max_iter), float(tol))
    dt = time.perf_counter() - t0
    V = np.array(s.get_V(), dtype=complex, copy=True)
    return V, bool(ok), int(s.get_nb_iter()), dt


def dc_angles(d):
    # DC power flow on the current topology (series susceptances, no shunts, no phase shifts). Returns angles in degrees.
    nb, on = d["nb"], d["on"]
    f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
    ref = np.nonzero(d["REF"])[0]
    keep = np.setdiff1d(np.arange(nb), ref)
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    th = np.zeros(nb)
    th[keep] = splu(B[keep][:, keep]).solve(P)
    return np.degrees(th) + float(d["Va"][ref[0]])


def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


def pick_outages(d, extra, mode, rng):
    # Indices (into the scenario's branch rows) of `extra` more lines to remove, keeping the grid in one piece.
    # mode "clustered": the lines nearest (in hops) to a random centre bus, like a storm footprint.
    # mode "uniform": random lines anywhere, like the released test sets.
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    hop = None
    if mode == "clustered":
        center = int(rng.integers(nb))
        A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
        hop = dijkstra(A, directed=False, unweighted=True, indices=center)
        dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    else:
        order = rng.permutation(cand)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if hop is not None else np.nan
    return np.array(removed), radius


def solve_new_topology(d, removed, tol=1e-8, max_iter=30):
    # Exact AC solution after removing `removed`; starts: the base solution, then flat, then DC.
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][removed] = False
    Y, S, ref, pv, pq = system_of(d2)
    base = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    flat = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(np.where(d2["REF"], d2["Va"], 0.0)))
    d2["Va_dc"] = dc_angles(d2)
    dc = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(d2["Va_dc"]))
    for name, V0 in [("base", base), ("flat", flat), ("dc", dc)]:
        if HAVE_KLU:
            V, ok, it, _ = klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol)
        else:
            V, it, norms, _ = newton(Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
            ok = norms[-1] <= tol
        if ok:
            d2["Vm"], d2["Va"] = np.abs(V), np.degrees(np.angle(V))
            d2["solved_from"] = name
            return d2
    return None




"""J1 core: true AC power-flow voltage sensitivities from the Jacobian, a finite-difference check of them,
hop distances in the grid graph, and the comparison metrics. Uses build_ybus / dsbus_dv / newton / SN_MVA from
the step-1/6 core."""
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


def ac_system(d):
    """Ybus, true complex voltages, specified injections (p.u.) and bus sets of one stored scenario."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    V = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    return Y, V, S, pv, pq


def power_flow_jacobian(Y, V, pv, pq):
    """Jacobian of [P mismatch at pv+pq; Q mismatch at pq] with respect to [angle at pv+pq; magnitude at pq]."""
    pvpq = np.r_[pv, pq]
    dS_dVm, dS_dVa = dsbus_dv(Y, V)
    return sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                      sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc"), pvpq


def true_sensitivity_rows(d, targets):
    """For each target bus (PQ in this scenario): dVm_target / dPd_j and dVm_target / dQd_j for every bus j,
    in p.u. voltage per p.u. load (100 MW or 100 Mvar). A load increase is a lower injection, hence the minus sign.
    Loads at the slack bus (P) and at voltage-controlled buses (Q) do not move any voltage: those entries are 0."""
    Y, V, S, pv, pq = ac_system(d)
    J, pvpq = power_flow_jacobian(Y, V, pv, pq)
    lu = splu(J.T.tocsc())
    n1, nb = len(pvpq), d["nb"]
    pos_pq = {int(b): n1 + i for i, b in enumerate(pq)}
    out = {}
    for t in targets:
        if int(t) not in pos_pq:
            continue
        e = np.zeros(J.shape[0]); e[pos_pq[int(t)]] = 1.0
        y = lu.solve(e)
        TP, TQ = np.zeros(nb), np.zeros(nb)
        TP[pvpq] = -y[:n1]
        TQ[pq] = -y[n1:]
        out[int(t)] = (TP, TQ)
    return out


def ac_fd_check(d, t, delta_pu=0.01):
    """Finite-difference check of the Jacobian sensitivities. Converge the base case, raise the reactive load at
    bus t by delta_pu (1 Mvar for 0.01), converge again, and compare dVm_t / dQd_t from the two solutions with the
    Jacobian value at the base solution. Returns (finite difference, Jacobian)."""
    Y, V, S, pv, pq = ac_system(d)
    V0, _, n0, _ = newton(Y, S, V, pv, pq, tol=1e-11, max_it=20)
    S2 = S.copy(); S2[t] -= 1j * delta_pu
    V2, _, n2, _ = newton(Y, S2, V0, pv, pq, tol=1e-11, max_it=20)
    if not n0 or not n2 or n0[-1] > 1e-9 or n2[-1] > 1e-9:
        return np.nan, np.nan
    J, pvpq = power_flow_jacobian(Y, V0, pv, pq)
    row = np.nonzero(pq == t)[0]
    e = np.zeros(J.shape[0]); e[len(pvpq) + int(row[0])] = 1.0
    jac = float(-splu(J.T.tocsc()).solve(e)[len(pvpq) + int(row[0])])
    return float((np.abs(V2[t]) - np.abs(V0[t])) / delta_pu), jac


def hop_distances(d, targets):
    """Number of in-service lines between each target and every bus (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    f, t = d["f"][on], d["t"][on]
    A = sp.csr_matrix((np.ones(2 * len(f)), (np.r_[f, t], np.r_[t, f])), shape=(nb, nb))
    return shortest_path(A, unweighted=True, directed=False, indices=np.asarray(targets, dtype=np.int64))


def row_metrics(gP, gQ, tP, tQ, pq_mask, pv_mask):
    """Compare GENCO's sensitivity row with the true one (same units). Only loads that can move voltages enter the
    gain and cosine: P at every non-slack bus, Q at PQ buses. Leakage: share of GENCO's reactive-load sensitivity that
    sits on voltage-controlled buses, where the true value is exactly 0."""
    g = np.r_[gP[pq_mask | pv_mask], gQ[pq_mask]]
    tr = np.r_[tP[pq_mask | pv_mask], tQ[pq_mask]]
    tt = float(tr @ tr)
    gain = float(g @ tr / tt) if tt > 0 else np.nan
    cos = float(g @ tr / (np.linalg.norm(g) * np.sqrt(tt))) if tt > 0 and np.linalg.norm(g) > 0 else np.nan
    rel_err = float(np.linalg.norm(g - tr) / np.sqrt(tt)) if tt > 0 else np.nan
    leak = float(np.abs(gQ[~pq_mask]).sum() / max(np.abs(gQ).sum(), 1e-30))
    gq, tq = gQ[pq_mask], tQ[pq_mask]
    tqq = float(tq @ tq)
    gain_q = float(gq @ tq / tqq) if tqq > 0 else np.nan
    cos_q = float(gq @ tq / (np.linalg.norm(gq) * np.sqrt(tqq))) if tqq > 0 and np.linalg.norm(gq) > 0 else np.nan
    return {"gain": gain, "cosine": cos, "relative_error": rel_err, "gain_Q": gain_q, "cosine_Q": cos_q, "pv_q_leakage": leak}


def hop_profile(abs_row, hops, max_hop=8):
    """Sum of |sensitivity| at each hop distance 0..max_hop (the last bin holds max_hop and beyond)."""
    h = np.where(np.isfinite(hops), np.minimum(hops, max_hop), max_hop).astype(int)
    return np.bincount(h, weights=abs_row, minlength=max_hop + 1)[: max_hop + 1]


def slope_through(x, y):
    """Least-squares slope of y on x (both centred): how strongly GENCO's voltage follows the true one."""
    x = np.asarray(x, float) - np.mean(x); y = np.asarray(y, float) - np.mean(y)
    return float(x @ y / (x @ x)) if x @ x > 0 else np.nan


"""K1 core: one storm topology with every load and every non-slack generator scaled by a load factor lambda (the
slack takes the rest), the AC truth along lambda up to the point where the AC equations stop having a solution (the
"nose"), a signature that the nose is genuine, and the screening decisions GENCO, DC and the Newton check make at
every lambda. Uses build_ybus / dsbus_dv / newton / newton_trace / klu_solve / system_of / dc_angles / SN_MVA /
HAVE_KLU from the step-1/6 core and power_flow_jacobian from the J1 core."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import shortest_path
from scipy.sparse.linalg import splu


class LambdaCase:
    """One topology; injections scaled by lambda: P at every non-slack bus = lambda * (Pg - Pd), Q at PQ buses =
    Qg - lambda * Qd (constant power factor), generator voltage set-points unchanged, the slack takes the rest."""

    def __init__(self, d):
        self.d = d
        self.Y, _, self.ref, self.pv, self.pq = system_of(d)
        self.nb = d["nb"]
        self.PQ, self.PV, self.REF = d["PQ"], d["PV"], d["REF"]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.P0 = (d["Pg"] - d["Pd"]) / SN_MVA
        self.Qg, self.Qd = d["Qg"] / SN_MVA, d["Qd"] / SN_MVA
        self.rated = d["on"] & (d["rate"] > 0)
        self.rate = d["rate"][self.rated]
        self.vmin, self.vmax = d["min_vm_pu"], d["max_vm_pu"]
        Vm_set = np.where(d["PQ"], 1.0, d["Vm"])
        self.flat = Vm_set * np.exp(1j * np.radians(np.where(d["REF"], d["Va"], 0.0)))
        self.dc = Vm_set * np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"])))
        self.ld_dc1 = self.loading(np.exp(1j * np.radians(np.nan_to_num(d["Va_dc"]))))   # DC state, AC branch equations

    # ---------------------------------------------------------------- physics at a given lambda
    def S(self, lam):
        return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson from V0 at this lambda. Returns V, converged, iterations, seconds."""
        S = self.S(lam)
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, self.pv, self.pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, self.pv, self.pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve_any(self, lam, starts, max_iter=30, tol=1e-8):
        """First converged solution over a list of (name, V0) starts; (None, None) if none converges."""
        for name, V0 in starts:
            V, ok, _, _ = self.solve(lam, V0, max_iter, tol)
            if ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2:
                return V, name
        return None, None

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / self.rate

    def dc_loading(self, lam):
        """DC power flow at this lambda: DC angles scale with lambda, flows through the AC branch equations."""
        return self.loading(np.exp(1j * np.radians(lam * np.nan_to_num(self.d["Va_dc"]))))

    def mismatch(self, lam, V):
        """Active mismatch (MW) at non-slack buses and reactive mismatch (Mvar) at PQ buses of a voltage state."""
        S = (V * np.conj(self.Y @ V) - self.S(lam)) * SN_MVA
        return np.abs(S.real[self.nonref]), np.abs(S.imag[self.pq])

    def jacobian(self, V):
        return power_flow_jacobian(self.Y, V, self.pv, self.pq)[0]

    # ---------------------------------------------------------------- what a screen would say about a state
    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        low = self.PQ & (Vm < self.vmin + margin_v)
        high = self.PQ & (Vm > self.vmax - margin_v)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        return {"min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA)}


# ---------------------------------------------------------------- the nose
def find_lambda_max(case, V_at_one=None, step=0.02, width=0.0025, lam_min=0.5, lam_cap=2.0, tol=1e-8, max_iter=30):
    """Largest load factor with an AC solution, by marching up from a solvable point and bisecting the first failure.
    At every failure the flat and DC starts are tried too, so a failure of the continuation start alone does not count.
    Returns a dict: lam_max, V_max (solution there), lam_first (first solvable point found), path [(lam, who)],
    n_solves, capped (True if solvable up to lam_cap)."""
    path, n = [], 0
    lam, V = 1.0, V_at_one
    if V is None:                       # the storm has no solution at lambda = 1: go down until one appears
        lam = 1.0 - step
        while lam >= lam_min:
            n += 3
            V, who = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc), ("base", case.state(case.d["Vm"], case.d["Va"]))], max_iter, tol)
            if V is not None:
                path.append((lam, who)); break
            lam = round(lam - step, 6)
        if V is None:
            return {"lam_max": np.nan, "V_max": None, "lam_first": np.nan, "path": path, "n_solves": n, "capped": False}
    else:
        path.append((1.0, "storm"))
    lam_first = lam
    lo, V_lo = lam, V
    hi = None
    while hi is None:                   # march up
        trial = round(lo + step, 6)
        if trial > lam_cap:
            return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": True}
        n += 1
        V2, ok, _, _ = case.solve(trial, V_lo, max_iter, tol)
        if ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2:
            lo, V_lo = trial, V2; path.append((trial, "continuation")); continue
        n += 2
        V2, who = case.solve_any(trial, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
        if V2 is not None:
            lo, V_lo = trial, V2; path.append((trial, who)); continue
        hi = trial
    while hi - lo > width:              # bisect
        mid = round(0.5 * (lo + hi), 8)
        n += 1
        V2, ok, _, _ = case.solve(mid, V_lo, max_iter, tol)
        if not (ok and np.all(np.isfinite(V2)) and np.abs(V2).min() > 0.2):
            n += 2
            V2, who = case.solve_any(mid, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            lo, V_lo = mid, V2; path.append((mid, "bisection"))
        else:
            hi = mid
    return {"lam_max": lo, "V_max": V_lo, "lam_first": lam_first, "path": path, "n_solves": n, "capped": False}


def truth_below_nose(case, lam_max, V_max, lams, tol=1e-8, max_iter=30):
    """AC solutions at the given load factors (all <= lam_max), marching down from the solution at the nose.
    Returns {lam: V} (missing where Newton failed, which should not happen below the nose)."""
    out, V = {}, V_max
    for lam in sorted(lams, reverse=True):
        V2, ok, _, _ = case.solve(lam, V, max_iter, tol)
        if not ok:
            V2, _ = case.solve_any(lam, [("flat", case.flat), ("dc", case.dc)], max_iter, tol)
            ok = V2 is not None
        if ok:
            out[lam] = V2; V = V2
    return out


def sigma_min(J, iters=40, seed=0):
    """Smallest singular value of the Jacobian, 1 / largest singular value of its inverse, by power iteration on
    J^-1 J^-T with one LU factorisation of J and one of J^T. Drops toward 0 at a genuine nose."""
    J = J.tocsc()
    lu, luT = splu(J), splu(J.T.tocsc())
    x = np.random.default_rng(seed).standard_normal(J.shape[0])
    x /= np.linalg.norm(x)
    val = np.nan
    for _ in range(iters):
        y = lu.solve(luT.solve(x))
        val = float(np.linalg.norm(y))
        x = y / val
    return 1.0 / np.sqrt(val)


# ---------------------------------------------------------------- helpers
def hop_distance(d, a, b):
    """Number of in-service lines between buses a and b (inf if not connected)."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    return float(shortest_path(A, unweighted=True, directed=False, indices=int(a))[int(b)])


def auroc(score, positive):
    """Area under the ROC curve by ranks (ties get the average rank); nan if one class is empty."""
    score, positive = np.asarray(score, float), np.asarray(positive, bool)
    ok = np.isfinite(score); score, positive = score[ok], positive[ok]
    n1, n0 = positive.sum(), (~positive).sum()
    if n1 == 0 or n0 == 0:
        return np.nan
    order = np.argsort(score, kind="mergesort")
    ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
    s = score[order]
    i = 0
    while i < len(s):                   # average ranks of ties
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = 0.5 * (i + j) + 1
        i = j + 1
    return float((ranks[positive].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


"""N core (reviewer items 4, 5, 9): generator reactive limits inside the AC truth and the check; stress patterns other
than the K1 scaling (loads only, reactive demand only, area-to-area transfer, generator outage on top of the storm);
warm-start timing of the classical solver. Uses LambdaCase / find_lambda_max / truth_below_nose from the K1 core and
build_ybus / newton / klu_solve / system_of / dc_angles / SN_MVA / HAVE_KLU from the step-1/6 core."""
import time

import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import dijkstra
from scipy.sparse.linalg import splu


# ================================================================ reactive limits
def bus_q_limits(d, gen_rows):
    """Per-bus reactive limits (p.u.) from the generator table of one scenario: sums over in-service generators.
    gen_rows: DataFrame with bus, min_q_mvar, max_q_mvar, in_service, is_slack_gen. Buses without a generator get 0/0."""
    nb = d["nb"]
    qmin, qmax = np.zeros(nb), np.zeros(nb)
    g = gen_rows[gen_rows["in_service"].to_numpy() > 0.5] if "in_service" in gen_rows else gen_rows
    np.add.at(qmin, g["bus"].to_numpy().astype(int), g["min_q_mvar"].to_numpy(float) / SN_MVA)
    np.add.at(qmax, g["bus"].to_numpy().astype(int), g["max_q_mvar"].to_numpy(float) / SN_MVA)
    return qmin, qmax


def q_generation(Y, V, Qd_pu):
    """Reactive generation implied by a voltage state (p.u.): injected Q plus the reactive load."""
    return (V * np.conj(Y @ V)).imag + Qd_pu


class QLimCase(LambdaCase):
    """A LambdaCase whose solver enforces generator reactive limits: a PV bus whose generation would exceed its limit
    is held at the limit as a PQ bus (and released again if its voltage recovers past the set-point). The outer loop
    is the standard one used in power-flow programs; a case counts as solved only when no further switching is needed."""

    def __init__(self, d, qmin_pu, qmax_pu, max_outer=15, eps=1e-6):
        super().__init__(d)
        self.qmin, self.qmax, self.max_outer, self.eps = qmin_pu, qmax_pu, max_outer, eps
        self.vset = d["Vm"].copy()                      # generator set-points (the base solution holds them)
        self.limited = self.PV & (self.qmax >= self.qmin)
        self.last = {"switched": {}, "outer": 0, "iters": 0}

    def _inner(self, S, V0, pv, pq, max_iter, tol):
        if HAVE_KLU:
            return klu_solve(self.Y, S, V0, self.ref, pv, pq, max_iter, tol)
        t0 = time.perf_counter()
        V, it, norms, _ = newton(self.Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
        return V, bool(norms[-1] <= tol), it, time.perf_counter() - t0

    def solve(self, lam, V0, max_iter=30, tol=1e-8):
        """Newton-Raphson with reactive-limit switching. Returns V, converged, total inner iterations, seconds.
        `self.last` keeps the switched buses of the last call ({bus: 'max' | 'min'}) and the number of outer rounds."""
        t0 = time.perf_counter()
        S_base = self.S(lam)
        Qd = lam * self.Qd
        V = np.asarray(V0, complex).copy()
        switched, backs, total_it = {}, {}, 0
        for outer in range(1, self.max_outer + 1):
            sw = np.array(sorted(switched), dtype=int)
            pv = np.array([b for b in self.pv if b not in switched], dtype=int)
            pq = np.array(sorted(set(self.pq.tolist()) | set(switched)), dtype=int)
            S = S_base.copy()
            if len(sw):
                S[sw] = S_base.real[sw] + 1j * (np.array([self.qmax[b] if switched[b] == "max" else self.qmin[b] for b in sw]) - Qd[sw])
            V[pv] = self.vset[pv] * np.exp(1j * np.angle(V[pv]))      # PV buses start at their set-point
            V, ok, it, _ = self._inner(S, V, pv, pq, max_iter, tol)
            total_it += it
            if not (ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2):
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, False, total_it, time.perf_counter() - t0
            Qg = q_generation(self.Y, V, Qd)
            to_max = [b for b in pv if self.limited[b] and Qg[b] > self.qmax[b] + self.eps]
            to_min = [b for b in pv if self.limited[b] and Qg[b] < self.qmin[b] - self.eps]
            Vm = np.abs(V)
            back = [b for b in sw if backs.get(b, 0) < 2 and ((switched[b] == "max" and Vm[b] > self.vset[b] + self.eps) or
                                                             (switched[b] == "min" and Vm[b] < self.vset[b] - self.eps))]
            if not to_max and not to_min and not back:
                self.last = {"switched": dict(switched), "outer": outer, "iters": total_it}
                return V, True, total_it, time.perf_counter() - t0
            for b in to_max:
                switched[int(b)] = "max"
            for b in to_min:
                switched[int(b)] = "min"
            for b in back:
                del switched[int(b)]; backs[int(b)] = backs.get(int(b), 0) + 1
        self.last = {"switched": dict(switched), "outer": self.max_outer, "iters": total_it}
        return V, False, total_it, time.perf_counter() - t0

    def decisions(self, lam, V, margin_line=0.0, margin_v=0.0):
        """As LambdaCase.decisions, but voltage limits are checked at every non-slack bus (a generator at its limit no
        longer holds its voltage) and the reactive generation beyond the limits is reported."""
        Vm = np.abs(V)
        ld = self.loading(V)
        P, Q = self.mismatch(lam, V)
        chk = ~self.REF
        low = chk & (Vm < self.vmin + margin_v)
        high = chk & (Vm > self.vmax - margin_v)
        vm_chk = np.where(chk, Vm, np.inf)
        Qg = q_generation(self.Y, V, lam * self.Qd)
        over_q = self.limited & (Qg > self.qmax + 1e-6)
        return {"min_vm": float(vm_chk.min()), "weak_bus": int(np.argmin(vm_chk)),
                "n_low_v": int(low.sum()), "n_high_v": int(high.sum()), "n_over": int((ld >= 1.0 - margin_line).sum()),
                "max_loading": float(ld.max()) if len(ld) else np.nan,
                "resP_max_MW": float(P.max()), "resP_p99_MW": float(np.percentile(P, 99)), "resP_mean_MW": float(P.mean()),
                "resQ_max_Mvar": float(Q.max()) if len(Q) else np.nan,
                "mismatch_pu": float(max(P.max(), Q.max() if len(Q) else 0.0) / SN_MVA),
                "n_gen_over_qmax": int(over_q.sum()), "q_over_qmax_Mvar": float(SN_MVA * np.clip(Qg - self.qmax, 0, None)[self.limited].sum())}


# ================================================================ stress patterns
class StressCase(LambdaCase):
    """A storm topology stressed in one direction by a factor lambda (1 = the actual operating point):
    'all'      every load and every non-slack generator scaled together (the K1 pattern; slack takes the rest);
    'load'     loads only (P and Q at constant power factor); generation fixed, the slack takes the growth;
    'reactive' reactive demand only (Q of every load); active power unchanged;
    'transfer' loads inside an area scaled, generation outside the area scaled up by the same MW (a transfer into
               the storm area); the slack takes the losses."""

    def __init__(self, d, mode="all", area=None, kappa=None):
        super().__init__(d)
        self.mode = mode
        self.Pg, self.Pd = d["Pg"] / SN_MVA, d["Pd"] / SN_MVA
        self.area = np.zeros(self.nb, bool) if area is None else np.asarray(area, bool)
        self.genB = (self.Pg > 0) & ~self.area & ~self.REF
        self.kappa = kappa if kappa is not None else (self.Pd[self.area].sum() / max(self.Pg[self.genB].sum(), 1e-9))

    def S(self, lam):
        if self.mode == "all":
            return lam * self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "load":
            return (self.Pg - lam * self.Pd) + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "reactive":
            return self.P0 + 1j * (self.Qg - lam * self.Qd)
        if self.mode == "transfer":
            fl = 1.0 + (lam - 1.0) * self.area
            fg = 1.0 + (lam - 1.0) * self.kappa * self.genB
            return (self.Pg * fg - self.Pd * fl) + 1j * (self.Qg - self.Qd * fl)
        raise ValueError(self.mode)

    def dc_loading(self, lam):
        """DC loading at this lambda: angles from the scaled active injections through the DC susceptance matrix."""
        d = self.d
        nb, on = d["nb"], d["on"]
        f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
        B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
        keep = self.nonref
        th = np.zeros(nb)
        th[keep] = splu(B[keep][:, keep]).solve(self.S(lam).real[keep])
        return self.loading(np.exp(1j * (th + np.radians(float(d["Va"][self.ref[0]])))))


def storm_area(d, footprint_buses, min_load_share=0.10, max_hops=6):
    """Buses within h hops of the storm footprint, h grown until the area holds at least `min_load_share` of the load."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=np.asarray(footprint_buses, int), min_only=True)
    total = max(d["Pd"].sum(), 1e-9)
    for h in range(0, max_hops + 1):
        area = hop <= h
        if d["Pd"][area].sum() >= min_load_share * total:
            return area, h, hop
    return hop <= max_hops, max_hops, hop


def footprint_of(d, removed_rows):
    """Endpoint buses of the removed lines (branch-row indices)."""
    return np.unique(np.r_[d["f"][removed_rows], d["t"][removed_rows]])


def pick_generator_outage(d, gen_rows, hop, max_hops=3):
    """The largest in-service non-slack generator (by active output; by reactive range if none produces active power)
    whose bus lies within `max_hops` of the storm footprint; the largest anywhere if there is none. Returns its row
    label in gen_rows (or None)."""
    g = gen_rows
    keep = np.ones(len(g), bool)
    if "in_service" in g:
        keep &= g["in_service"].to_numpy(float) > 0.5
    if "is_slack_gen" in g:
        keep &= g["is_slack_gen"].to_numpy(float) < 0.5
    keep &= ~d["REF"][g["bus"].to_numpy().astype(int)]
    g = g[keep]
    if len(g) == 0:
        return None
    size = g["p_mw"].to_numpy(float)
    if not (size > 0).any():
        size = g["max_q_mvar"].to_numpy(float) - g["min_q_mvar"].to_numpy(float)
    near = hop[g["bus"].to_numpy().astype(int)] <= max_hops
    cand = g[near] if near.any() else g
    size_c = size[near] if near.any() else size
    return int(cand.index[int(np.argmax(size_c))])


def remove_generator(d, gen_rows, row):
    """The truth model after one generator is lost: its P and Q leave the bus (the slack takes the P); if no in-service
    generator remains at the bus, it becomes a PQ bus. Returns the new dict and the reduced generator table."""
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    g = gen_rows.copy()
    bus = int(g.loc[row, "bus"])
    d2["Pg"][bus] -= float(g.loc[row, "p_mw"]); d2["Qg"][bus] -= float(g.loc[row, "q_mvar"])
    g = g.drop(index=row)
    left = g[(g["bus"].to_numpy().astype(int) == bus) & (g["in_service"].to_numpy() > 0.5)] if "in_service" in g else g[g["bus"] == bus]
    if len(left) == 0 and not d2["REF"][bus]:
        d2["PV"][bus] = False; d2["PQ"][bus] = True
        d2["Pg"][bus] = 0.0; d2["Qg"][bus] = 0.0
    return d2, g, bus


# ================================================================ warm starts (item 9)
def timed_solves(case, lam, starts, max_iter=30, tol=1e-8, reps=3):
    """Iterations and the median time (ms) of the solver from each start; converged flag per start."""
    out = {}
    for name, V0 in starts:
        its, ts, oks = [], [], []
        for _ in range(reps):
            V, ok, it, dt = case.solve(lam, V0, max_iter, tol)
            its.append(it); ts.append(1000 * dt); oks.append(bool(ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2))
        out[name] = {"converged": all(oks), "iters": int(np.median(its)), "ms": float(np.median(ts))}
    return out


"""V core: an independent implementation of the AC power-flow equations for validation. PYPOWER (the Python port of
MATPOWER) builds its own admittance matrix from the raw branch parameters of the released tables (r, x, b, tap, shift,
status) and the bus shunts (GS, BS), its own injections from the bus loads and the generator table, and solves with its
own Newton-Raphson (and its own reactive-limit loop). Nothing of our custom Ybus (built from the stored Yff/Yft/Ytf/Ytt
entries), our Python Newton or lightsim2grid is used here, so an agreement between the two is evidence against a shared
modelling error. Uses read_cols / find_local / SN_MVA from the step-1 core."""
import time

import numpy as np
import pandas as pd
from pypower.api import ppoption, runpf
from pypower.bustypes import bustypes
from pypower.dcpf import dcpf
from pypower.makeBdc import makeBdc
from pypower.makeSbus import makeSbus
from pypower.makeYbus import makeYbus
from pypower.newtonpf import newtonpf
from pypower.idx_bus import BUS_I, BUS_TYPE, PD, QD, GS, BS, BUS_AREA, VM, VA, BASE_KV, ZONE, VMAX, VMIN
from pypower.idx_gen import GEN_BUS, PG, QG, QMAX, QMIN, VG, MBASE, GEN_STATUS, PMAX, PMIN
from pypower.idx_brch import F_BUS, T_BUS, BR_R, BR_X, BR_B, RATE_A, RATE_B, RATE_C, TAP, SHIFT, BR_STATUS, ANGMIN, ANGMAX

RAW_BUS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "vn_kv", "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc"]
RAW_BR = ["scenario", "idx", "from_bus", "to_bus", "r", "x", "b", "tap", "shift", "rate_a", "br_status", "ang_min", "ang_max",
          "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i", "Ytt_r", "Ytt_i"]
RAW_GEN = ["scenario", "idx", "bus", "p_mw", "q_mvar", "min_p_mw", "max_p_mw", "min_q_mvar", "max_q_mvar", "in_service", "is_slack_gen"]
PP_OPT = ppoption(VERBOSE=0, OUT_ALL=0, PF_ALG=1)


RAW_CACHE = {}


def preload_raw(local_dir, level, ids):
    """Read the raw tables of many scenarios of one level at once (one pass over the parquet files) into RAW_CACHE."""
    ids = np.unique(np.asarray(ids, dtype=np.int64))
    have = RAW_CACHE.get(level)
    if have is not None:
        ids = ids[~np.isin(ids, have["ids"])]
        if len(ids) == 0:
            return
    bus = read_cols(find_local(local_dir, level, "bus_data"), RAW_BUS, scen=ids)
    br = read_cols(find_local(local_dir, level, "branch_data"), RAW_BR, scen=ids)
    gen = read_cols(find_local(local_dir, level, "gen_data"), RAW_GEN, scen=ids)
    for t in (bus, br, gen):
        t["scenario"] = t["scenario"].astype(np.int64)
    if have is None:
        RAW_CACHE[level] = {"ids": ids, "bus": bus, "br": br, "gen": gen}
    else:
        RAW_CACHE[level] = {"ids": np.r_[have["ids"], ids], "bus": pd.concat([have["bus"], bus]), "br": pd.concat([have["br"], br]), "gen": pd.concat([have["gen"], gen])}
    c = RAW_CACHE[level]
    c["g_bus"], c["g_br"], c["g_gen"] = c["bus"].groupby("scenario"), c["br"].groupby("scenario"), c["gen"].groupby("scenario")


def raw_tables(local_dir, level, sid):
    """The released bus, branch and generator tables of one scenario, with the raw network parameters."""
    sid = int(sid)
    c = RAW_CACHE.get(level)
    if c is not None and sid in c["g_bus"].groups:
        bus, br, gen = (c[g].get_group(sid).copy() for g in ["g_bus", "g_br", "g_gen"])
    else:
        bus = read_cols(find_local(local_dir, level, "bus_data"), RAW_BUS, scen=[sid])
        br = read_cols(find_local(local_dir, level, "branch_data"), RAW_BR, scen=[sid])
        gen = read_cols(find_local(local_dir, level, "gen_data"), RAW_GEN, scen=[sid])
    bus = bus.sort_values("bus").reset_index(drop=True); br = br.sort_values("idx").reset_index(drop=True); gen = gen.sort_values("idx").reset_index(drop=True)
    if len(bus) == 0 or len(br) == 0 or len(gen) == 0:
        raise RuntimeError(f"scenario {sid} of level {level} is not on disk")
    return bus, br, gen


def admittance_check(br):
    """Recompute the branch admittance entries from r, x, b, tap and shift with the MATPOWER formulas and compare with the
    stored Yff/Yft/Ytf/Ytt: a check on the units, the tap and phase-shift conventions and the branch status."""
    on = br["br_status"].to_numpy(float) > 0.5
    ys = on / (br["r"].to_numpy(float) + 1j * br["x"].to_numpy(float))
    bc = on * br["b"].to_numpy(float)
    tap = br["tap"].to_numpy(float).copy(); tap[tap == 0] = 1.0
    t = tap * np.exp(1j * np.radians(br["shift"].to_numpy(float)))
    Ytt = ys + 1j * bc / 2
    Yff = Ytt / (t * np.conj(t))
    Yft = -ys / np.conj(t)
    Ytf = -ys / t
    stored = {k: br[k + "_r"].to_numpy(float) + 1j * br[k + "_i"].to_numpy(float) for k in ["Yff", "Yft", "Ytf", "Ytt"]}
    diff = {k: float(np.max(np.abs(v - stored[k]))) for k, v in [("Yff", Yff), ("Yft", Yft), ("Ytf", Ytf), ("Ytt", Ytt)]}
    scale = float(np.max(np.abs(stored["Yff"])))
    return {"max_abs_diff": max(diff.values()), "max_rel_diff": max(diff.values()) / scale, "per_entry": diff,
            "branches": int(len(br)), "in_service": int(on.sum()), "transformers_tap_not_1": int((tap != 1.0).sum()),
            "phase_shifters": int((br["shift"].to_numpy(float) != 0).sum()), "zero_impedance": int((np.abs(br["r"] + 1j * br["x"]) == 0).sum())}


class Indep:
    """One scenario's tables as a PYPOWER case. Lines are removed by their idx; the load factor lam scales every load and
    every non-slack generator, as our LambdaCase does (the slack takes the rest); generator voltage set-points are the
    released Vm of their buses and are never changed."""

    def __init__(self, bus, br, gen):
        self.bus, self.br, self.gen = bus.reset_index(drop=True), br.reset_index(drop=True), gen.reset_index(drop=True)
        self.nb = len(self.bus)
        self.PQ, self.PV, self.REF = (self.bus[c].to_numpy(float) > 0.5 for c in ["PQ", "PV", "REF"])
        self.vmin, self.vmax = self.bus["min_vm_pu"].to_numpy(float), self.bus["max_vm_pu"].to_numpy(float)
        self.idx = self.br["idx"].to_numpy().astype(int)
        self.rate = self.br["rate_a"].to_numpy(float)

    # ---------------------------------------------------------------- the case
    def ppc(self, removed_idx=(), lam=1.0, Vm0=None, Va0=None):
        b, g, r = self.bus, self.gen, self.br
        nb, ng, nl = len(b), len(g), len(r)
        bus = np.zeros((nb, 13))
        bus[:, BUS_I] = np.arange(nb)                                   # internal numbering: our buses are already 0..nb-1
        bus[:, BUS_TYPE] = np.where(self.REF, 3, np.where(self.PV, 2, 1))
        bus[:, PD] = lam * b["Pd"].to_numpy(float); bus[:, QD] = lam * b["Qd"].to_numpy(float)
        bus[:, GS] = b["GS"].to_numpy(float) * SN_MVA; bus[:, BS] = b["BS"].to_numpy(float) * SN_MVA   # stored in p.u. (datakit divides by baseMVA)
        bus[:, BUS_AREA] = 1; bus[:, ZONE] = 1
        bus[:, VM] = b["Vm"].to_numpy(float) if Vm0 is None else np.asarray(Vm0, float)
        bus[:, VA] = b["Va"].to_numpy(float) if Va0 is None else np.asarray(Va0, float)
        bus[:, BASE_KV] = b["vn_kv"].to_numpy(float); bus[:, VMAX] = self.vmax; bus[:, VMIN] = self.vmin
        # PV/REF buses of the released tables without an in-service generator (datakit turns such buses into PQ, so the released
        # Texas data should have none; counted in the conventions check): a dummy generator holds their voltage as the flags say,
        # so that PYPOWER models the same bus types as our solver, which trusts the flags
        on = g["in_service"].to_numpy(float) > 0.5
        has = np.zeros(nb, bool); has[g["bus"].to_numpy().astype(int)[on]] = True
        dummy = np.nonzero((self.PV | self.REF) & ~has)[0]
        ng2 = ng + len(dummy)
        gen = np.zeros((ng2, 21))
        gb = np.r_[g["bus"].to_numpy().astype(int), dummy]
        gen[:, GEN_BUS] = gb
        slack = np.r_[g["is_slack_gen"].to_numpy(float) > 0.5, self.REF[dummy]]
        gen[:, PG] = np.r_[g["p_mw"].to_numpy(float), np.zeros(len(dummy))] * np.where(slack, 1.0, lam)
        gen[:, QG] = np.r_[g["q_mvar"].to_numpy(float), b["Qg"].to_numpy(float)[dummy]]
        gen[:, QMAX] = np.r_[g["max_q_mvar"].to_numpy(float), np.full(len(dummy), 1e9)]
        gen[:, QMIN] = np.r_[g["min_q_mvar"].to_numpy(float), np.full(len(dummy), -1e9)]
        gen[:, VG] = b["Vm"].to_numpy(float)[gb]                        # the set-point: the released voltage of the bus
        gen[:, MBASE] = SN_MVA
        gen[:, GEN_STATUS] = np.r_[on, np.ones(len(dummy), bool)].astype(float)
        gen[:, PMAX] = np.r_[g["max_p_mw"].to_numpy(float), np.zeros(len(dummy))]; gen[:, PMIN] = np.r_[g["min_p_mw"].to_numpy(float), np.zeros(len(dummy))]
        self.n_dummy = len(dummy)
        branch = np.zeros((nl, 17))                                     # 13 inputs + 4 flow columns that pfsoln fills
        branch[:, F_BUS] = r["from_bus"].to_numpy().astype(int); branch[:, T_BUS] = r["to_bus"].to_numpy().astype(int)
        branch[:, BR_R] = r["r"].to_numpy(float); branch[:, BR_X] = r["x"].to_numpy(float); branch[:, BR_B] = r["b"].to_numpy(float)
        branch[:, RATE_A] = self.rate
        tap = r["tap"].to_numpy(float).copy(); tap[tap == 0] = 1.0
        branch[:, TAP] = tap; branch[:, SHIFT] = r["shift"].to_numpy(float)
        status = r["br_status"].to_numpy(float) > 0.5
        if len(removed_idx):
            status = status & ~np.isin(self.idx, np.asarray(removed_idx, dtype=int))
        branch[:, BR_STATUS] = status.astype(float)
        branch[:, ANGMIN] = r["ang_min"].to_numpy(float); branch[:, ANGMAX] = r["ang_max"].to_numpy(float)
        return {"version": "2", "baseMVA": SN_MVA, "bus": bus, "gen": gen, "branch": branch}

    # ---------------------------------------------------------------- starts
    def start_state(self, ppc, start):
        """Initial complex voltage for a start: 'base' (the released state), 'flat', 'dc' (PYPOWER's DC power flow), or a
        (Vm, Va_deg) pair. Generator buses always begin at their set-point magnitude, as in PYPOWER's runpf."""
        bus, gen = ppc["bus"], ppc["gen"]
        ref, pv, pq = bustypes(bus, gen)
        if isinstance(start, str):
            if start == "base":
                Vm, Va = self.bus["Vm"].to_numpy(float), np.radians(self.bus["Va"].to_numpy(float))
            elif start == "flat":
                Vm, Va = np.ones(self.nb), np.radians(np.where(self.REF, self.bus["Va"].to_numpy(float), 0.0))
            elif start == "dc":
                Bbus, Bf, Pbusinj, Pfinj = makeBdc(ppc["baseMVA"], bus, ppc["branch"])
                Pbus = makeSbus(ppc["baseMVA"], bus, gen).real - Pbusinj - bus[:, GS] / ppc["baseMVA"]
                Va = dcpf(Bbus, Pbus, np.radians(np.where(self.REF, self.bus["Va"].to_numpy(float), 0.0)), ref, pv, pq)
                Vm = np.ones(self.nb)
            else:
                raise ValueError(start)
        else:
            Vm, Va = np.asarray(start[0], float), np.radians(np.asarray(start[1], float))
        V0 = Vm * np.exp(1j * Va)
        on = np.nonzero(gen[:, GEN_STATUS] > 0)[0]
        gbus = gen[on, GEN_BUS].astype(int)
        vcb = np.ones(self.nb); vcb[pq] = 0
        k = np.nonzero(vcb[gbus])[0]
        V0[gbus[k]] = gen[on[k], VG] / np.abs(V0[gbus[k]]) * V0[gbus[k]]
        return V0, ref, pv, pq

    # ---------------------------------------------------------------- solves
    def mismatch(self, ppc, V):
        """PYPOWER's power-flow mismatch (largest entry, p.u.) of a voltage state: P at every non-slack bus, Q at PQ buses."""
        bus, gen = ppc["bus"], ppc["gen"]
        ref, pv, pq = bustypes(bus, gen)
        Ybus, _, _ = makeYbus(ppc["baseMVA"], bus, ppc["branch"])
        Sbus = makeSbus(ppc["baseMVA"], bus, gen)
        mis = V * np.conj(Ybus @ V) - Sbus
        F = np.r_[mis[pv].real, mis[pq].real, mis[pq].imag]
        return float(np.max(np.abs(F))) if len(F) else 0.0

    def solve(self, removed_idx=(), lam=1.0, start="base", max_it=30, tol=1e-8):
        """PYPOWER's Newton-Raphson on the case. Returns a dict: converged, iterations, seconds, mismatch at the start and at
        the end (p.u.), the state (Vm, Va_deg) and the screening decisions made on it."""
        ppc = self.ppc(removed_idx, lam)
        V0, ref, pv, pq = self.start_state(ppc, start)
        opt = PP_OPT.copy(); opt["PF_TOL"], opt["PF_MAX_IT"] = tol, int(max_it)
        Ybus, Yf, Yt = makeYbus(ppc["baseMVA"], ppc["bus"], ppc["branch"])
        Sbus = makeSbus(ppc["baseMVA"], ppc["bus"], ppc["gen"])
        m0 = self.mismatch(ppc, V0)
        t0 = time.perf_counter()
        note = ""
        try:
            V, ok, it = newtonpf(Ybus, Sbus, V0, ref, pv, pq, opt)
        except Exception as exc:                      # a singular Jacobian on a diverging path makes SuperLU raise: that is a failed start
            V, ok, it = np.full(len(V0), np.nan, dtype=complex), False, int(max_it)
            note = f"solver error: {type(exc).__name__}"
        sec = time.perf_counter() - t0
        ok = bool(ok) and bool(np.all(np.isfinite(V)))
        out = {"converged": ok, "iterations": int(it), "seconds": sec, "mismatch_start_pu": m0, "note": note,
               "mismatch_end_pu": self.mismatch(ppc, V) if np.all(np.isfinite(V)) else np.inf}
        out.update(self.decisions(ppc, Yf, Yt, V, Ybus, Sbus, lam))
        return out

    def decisions(self, ppc, Yf, Yt, V, Ybus, Sbus, lam):
        if not np.all(np.isfinite(V)):
            nb = len(V)
            return {"Vm": np.full(nb, np.nan), "Va_deg": np.full(nb, np.nan), "loading": np.full(len(ppc["branch"]), np.nan), "overloaded_idx": [],
                    "n_over": 0, "max_loading": np.nan, "min_vm": np.nan, "weak_bus": -1, "n_low_v": 0, "n_high_v": 0, "low_v_buses": [], "Qg_bus_Mvar": np.full(nb, np.nan)}
        branch = ppc["branch"]
        f, t = branch[:, F_BUS].astype(int), branch[:, T_BUS].astype(int)
        Sf = V[f] * np.conj(Yf @ V) * ppc["baseMVA"]
        St = V[t] * np.conj(Yt @ V) * ppc["baseMVA"]
        rated = (branch[:, BR_STATUS] > 0) & (branch[:, RATE_A] > 0)
        ld = np.full(len(branch), np.nan); ld[rated] = np.maximum(np.abs(Sf), np.abs(St))[rated] / branch[rated, RATE_A]
        Vm = np.abs(V)
        pq_vm = np.where(self.PQ, Vm, np.inf)
        inj = V * np.conj(Ybus @ V)
        Qg_bus = (inj.imag + lam * self.bus["Qd"].to_numpy(float) / ppc["baseMVA"]) * ppc["baseMVA"]
        return {"Vm": Vm, "Va_deg": np.degrees(np.angle(V)), "loading": ld, "overloaded_idx": self.idx[rated & (ld >= 1.0)].tolist(),
                "n_over": int(np.nansum(ld >= 1.0)), "max_loading": float(np.nanmax(ld)) if rated.any() else np.nan,
                "min_vm": float(pq_vm.min()), "weak_bus": int(np.argmin(pq_vm)),
                "n_low_v": int((self.PQ & (Vm < self.vmin)).sum()), "n_high_v": int((self.PQ & (Vm > self.vmax)).sum()),
                "low_v_buses": np.nonzero(self.PQ & ((Vm < self.vmin) | (Vm > self.vmax)))[0].tolist(),
                "Qg_bus_Mvar": Qg_bus}

    def solve_qlim(self, removed_idx=(), lam=1.0, start="base", max_it=30, tol=1e-8, slack_unlimited=True, convention="matpower",
                   max_rounds=15, eps=1e-6):
        """Reactive-limit enforcement around PYPOWER's Newton-Raphson (PYPOWER's own runpf loop fails on current numpy, so the
        outer loop is written here with PYPOWER's primitives). Two conventions:
          'matpower'  as MATPOWER/PYPOWER: a generator beyond a limit is fixed at that limit and its bus becomes PQ (the other
                      generators at the bus keep the reactive output of the last solution); nothing is converted back;
          'release'   as our notebook N1: limits summed per bus; a PV bus whose total exceeds a limit is held at the sum as a
                      PQ bus, and released again when its voltage recovers past the set-point (at most twice per bus).
        slack_unlimited leaves the slack generator unlimited (as in N1); with False the slack bus is limited like the others and
        a PV bus takes over as slack when it is converted. Returns converged, rounds, the state, the decisions, the buses held at
        a limit and the generators' reactive outputs."""
        from pypower.pfsoln import pfsoln
        ppc = self.ppc(removed_idx, lam)
        bus, gen, branch = ppc["bus"].copy(), ppc["gen"].copy(), ppc["branch"].copy()
        V, ref, pv, pq = self.start_state(ppc, start)
        if slack_unlimited:
            s_ = np.r_[self.gen["is_slack_gen"].to_numpy(float) > 0.5, np.zeros(len(gen) - len(self.gen), bool)]
            gen[s_, QMAX], gen[s_, QMIN] = 1e9, -1e9
        opt = PP_OPT.copy(); opt["PF_TOL"], opt["PF_MAX_IT"] = tol, int(max_it)
        Ybus, Yf, Yt = makeYbus(ppc["baseMVA"], bus, branch)
        on = gen[:, GEN_STATUS] > 0
        gbus = gen[:, GEN_BUS].astype(int)
        qmin_bus, qmax_bus = np.zeros(self.nb), np.zeros(self.nb)
        np.add.at(qmin_bus, gbus[on], gen[on, QMIN]); np.add.at(qmax_bus, gbus[on], gen[on, QMAX])
        held = {}                       # bus -> 'max' | 'min'
        backs = {}
        t0 = time.perf_counter(); total_it = 0; ok = False; rounds = 0
        vset = gen[:, VG].copy()
        bus_types0 = bus[:, BUS_TYPE].copy()
        for rounds in range(1, max_rounds + 1):
            bus[:, BUS_TYPE] = bus_types0
            for b_ in held:
                bus[b_, BUS_TYPE] = 1
            ref, pv, pq = bustypes(bus, gen)
            if len(ref) == 0:
                break
            # fixed injections of the held buses: generation at the summed limit (the gens' QG entries sum to it)
            for b_, side in held.items():
                gi = np.nonzero(on & (gbus == b_))[0]
                lim = gen[gi, QMAX] if side == "max" else gen[gi, QMIN]
                gen[gi, QG] = lim
            Sbus = makeSbus(ppc["baseMVA"], bus, gen)
            # PV buses start at their set-point magnitude
            for b_ in pv:
                gi = np.nonzero(on & (gbus == b_))[0]
                V[b_] = vset[gi[0]] * np.exp(1j * np.angle(V[b_]))
            try:
                V, ok, it = newtonpf(Ybus, Sbus, V, ref, pv, pq, opt)
            except Exception:
                V, ok, it = np.full(len(V), np.nan, dtype=complex), False, int(max_it)
            total_it += int(it)
            if not (ok and np.all(np.isfinite(V)) and np.abs(V).min() > 0.2):
                ok = False
                break
            bus_, gen_, branch_ = pfsoln(ppc["baseMVA"], bus.copy(), gen.copy(), branch.copy(), Ybus, Yf, Yt, V, ref, pv, pq)
            Qg_bus = np.zeros(self.nb); np.add.at(Qg_bus, gbus[on], gen_[on, QG])
            Vm = np.abs(V)
            if convention == "matpower":
                viol_max = on & (gen_[:, QG] > gen_[:, QMAX] + eps) & np.isin(gbus, pv)
                viol_min = on & (gen_[:, QG] < gen_[:, QMIN] - eps) & np.isin(gbus, pv)
                to_max = sorted(set(gbus[viol_max].tolist())); to_min = sorted(set(gbus[viol_min].tolist()) - set(to_max))
                back = []
            else:
                to_max = [b_ for b_ in pv if Qg_bus[b_] > qmax_bus[b_] + eps and qmax_bus[b_] >= qmin_bus[b_]]
                to_min = [b_ for b_ in pv if Qg_bus[b_] < qmin_bus[b_] - eps and qmax_bus[b_] >= qmin_bus[b_]]
                back = [b_ for b_, side in held.items() if backs.get(b_, 0) < 2 and
                        ((side == "max" and Vm[b_] > vset[np.nonzero(gbus == b_)[0][0]] + eps) or
                         (side == "min" and Vm[b_] < vset[np.nonzero(gbus == b_)[0][0]] - eps))]
            if not to_max and not to_min and not back:
                gen = gen_
                break
            ok = False
            for b_ in to_max:
                held[int(b_)] = "max"
            for b_ in to_min:
                held[int(b_)] = "min"
            for b_ in back:
                del held[int(b_)]; backs[int(b_)] = backs.get(int(b_), 0) + 1
            if convention == "matpower":
                # the generators of a converted bus that are not themselves beyond a limit keep their last reactive output
                for b_ in list(to_max) + list(to_min):
                    gi = np.nonzero(on & (gbus == b_))[0]
                    for g_ in gi:
                        if not (viol_max[g_] or viol_min[g_]):
                            gen[g_, QG] = gen_[g_, QG]
                            gen[g_, QMAX] = gen[g_, QMIN] = gen_[g_, QG]      # frozen at that value for the rest of the loop
        sec = time.perf_counter() - t0
        out = {"converged": bool(ok), "rounds": int(rounds), "inner_iterations": int(total_it), "seconds": sec}
        Sbus = makeSbus(ppc["baseMVA"], bus, gen)
        out.update(self.decisions(ppc, Yf, Yt, V, Ybus, Sbus, lam))
        out.update({"buses_at_limit": sorted(held), "held": dict(held), "Qg_gen_Mvar": gen[:, QG].copy(), "convention": convention})
        return out


def complementarity(bus_df, gen_df, Vm, Qg_bus_Mvar, eps_q=1e-3, eps_v=1e-6):
    """Reactive-limit complementarity of a state, per PV bus with limits (per-bus aggregate of its in-service generators):
    within limits -> the voltage must sit at its set-point; at Qmax -> voltage at or below the set-point; at Qmin -> at or
    above. Returns counts of buses in each condition and of violations."""
    g = gen_df[gen_df["in_service"].to_numpy(float) > 0.5]
    nb = len(bus_df)
    qmin, qmax = np.zeros(nb), np.zeros(nb)
    np.add.at(qmin, g["bus"].to_numpy().astype(int), g["min_q_mvar"].to_numpy(float))
    np.add.at(qmax, g["bus"].to_numpy().astype(int), g["max_q_mvar"].to_numpy(float))
    has = np.zeros(nb, bool); has[g["bus"].to_numpy().astype(int)] = True
    pv = (bus_df["PV"].to_numpy(float) > 0.5) & has & (qmax >= qmin)
    vset = bus_df["Vm"].to_numpy(float)
    q = np.asarray(Qg_bus_Mvar, float); v = np.asarray(Vm, float)
    inside = pv & (q < qmax - eps_q) & (q > qmin + eps_q)
    at_max = pv & (q >= qmax - eps_q); at_min = pv & (q <= qmin + eps_q)
    beyond = pv & ((q > qmax + eps_q) | (q < qmin - eps_q))
    bad = (inside & (np.abs(v - vset) > eps_v)) | (at_max & (v > vset + eps_v)) | (at_min & (v < vset - eps_v)) | beyond
    return {"pv_buses_with_limits": int(pv.sum()), "inside": int(inside.sum()), "at_qmax": int(at_max.sum()), "at_qmin": int(at_min.sum()),
            "beyond_limits": int(beyond.sum()), "complementarity_violations": int(bad.sum()), "violating_buses": np.nonzero(bad)[0].tolist()[:20]}


def same_state(Vm_a, Vm_b, tol=1e-4):
    return bool(np.max(np.abs(np.asarray(Vm_a) - np.asarray(Vm_b))) < tol)


def state_diff(Vm_a, Va_a, Vm_b, Va_b):
    dva = (np.asarray(Va_a) - np.asarray(Va_b) + 180) % 360 - 180
    return {"max_dVm_pu": float(np.max(np.abs(np.asarray(Vm_a) - np.asarray(Vm_b)))), "max_dVa_deg": float(np.max(np.abs(dva)))}


# ---------------------------------------------------------------- notebook setup
import json, math, platform, zipfile, warnings, copy
import matplotlib.pyplot as plt
from IPython.display import display
from gridfm_graphkit.datasets.globals import PD_H, QD_H, PG_H

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)

# N-2 scenarios that are hard for GENCO: not used as storm base points, so the base points are ordinary
HARD2 = set(json.loads('[21, 22, 129, 131, 133, 158, 171, 436, 552, 555, 1231, 1251, 1264, 1282, 1294, 1396, 1446, 1449, 1457, 1521, 1574, 1578, 2212, 2222, 2288, 2353, 2390, 2394, 3314, 3379, 3851, 3852, 3900, 3903, 3905, 3927, 3928, 3947, 4622, 4628, 4698, 4711, 4722, 4723, 4762, 4777, 4778, 4779, 4798, 4801, 4851, 4853, 4878, 4896, 4898, 4899, 4902, 4920, 4923, 4924, 4961, 5231, 5258, 5368, 5382, 5949, 6025, 6057, 6208, 6230, 6294, 6319, 6320, 6356, 6367, 7259, 7261, 7356, 7658, 7684, 7689, 7844, 7863, 7870, 7882, 7968, 8188, 8255, 8411, 8487, 8595]'))
RAW3_PAT = re.compile(r"(?:^|/)(bus_data|branch_data|gen_data)\.parquet/scenario_partition=(\d+)/[^/]+$")
PART = 200


def read_all(path, scen):
    dset = ds.dataset(str(path), format="parquet", partitioning="hive")
    values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
    t = dset.to_table(filter=ds.field("scenario").isin(values)).to_pandas()
    return t.drop(columns=[c for c in ["scenario_partition"] if c in t.columns])


def pick_storm(d, extra, rng):
    """`extra` more lines to remove, the nearest (in hops) to a random centre bus, keeping the grid in one piece.
    Returns branch-row indices, the storm radius in hops and the centre bus; (None, None, centre) if impossible."""
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    center = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=center)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    order = cand[np.lexsort((rng.random(len(cand)), dist))]
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None, center
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]])))
    return np.array(removed), radius, center


print("ready | lightsim2grid (KLU) available:", HAVE_KLU)


def truth_dicts(level, ids):
    """Stored AC solutions of the given scenarios as compact dicts (the truth sits in the prediction slot too)."""
    ids = np.sort(np.asarray(ids, dtype=np.int64))
    bus = read_cols(find_local(LOCAL_DIR, level, "bus_data"), ["scenario", "bus", "Vm", "Va"], scen=ids)
    pred = pd.DataFrame({"scenario": bus.scenario.astype(np.int64), "bus": bus.bus.astype(np.int64),
                         "vm": bus.Vm, "va": np.radians(bus.Va)}).sort_values(["scenario", "bus"]).reset_index(drop=True)
    A = load_chunk(find_local(LOCAL_DIR, level, "bus_data"), find_local(LOCAL_DIR, level, "branch_data"), pred, ids)
    return {int(ids[i]): extract_scenario(A, i) for i in range(len(ids))}



import subprocess, math

## 1. The frozen protocol and the fine-tuned checkpoint

In [ ]:
T2_PROTOCOL = {
    "purpose": "the test of the fine-tuned GENCO (notebook T1) against the released model, paired on cases neither model was trained or selected on, "
               "and of a warning head that flags cases likely to be unresolved by the specified solver; fixed before any fine-tuned output is seen",
    "models": {"released": f"{REPO_ID}/{MODEL_DIR}", "fine_tuned": f"{T1_DIR}/genco_finetuned_state_dict.pt (SHA-256 checked against its summary)", "finetune_seed": FINETUNE_SEED},
    "verification": {"policy": POLICY, "note": "the frozen policy of F1 (no reactive limits, first converged start); every verdict comes from the solver"},
    "test_sets": {
        "F1": "the 870 contingencies of the frozen final evaluation (N-2 blocks 6 and 16), rebuilt from their stored base points and removed lines",
        "footprint": "a soft-footprint N-20 case per F1 base point (174), a construction held out of fine-tuning, seed rule default_rng(SEED*1000003 + base + 11)",
        "forgetting": f"released scenarios with released AC solutions: N-2 block 7 and N-10, N-20 block 0, the first {N_FORGET} of each",
        "near_fold": "the 120 K1 storms (N-2 blocks 0-1), their tracked-branch fold recomputed as in K1 and their AC truth along K1's load grid"},
    "primary_endpoints": [
        "1. clustered cases (F1 nearest/corridor families and footprint): decisions missed by GENCO alone per verified case, paired difference "
        "(fine-tuned - released), mean with a 95 % cluster bootstrap interval over base points; also cases with any missed decision (exact McNemar)",
        "2. worst-bus voltage error of GENCO alone on the same cases, paired difference, mean and median, cluster bootstrap",
        f"3. forgetting: worst-bus voltage error on the released scenarios, paired difference; non-inferior if the upper 95 % bound of the mean "
        f"increase is below {NONINF_MARGIN_PU} p.u. at each level",
        "4. near the fold: the lowest voltage at the fold (AC, released, fine-tuned) and GENCO's voltage drop over the last 5 % of load as a share of "
        "the AC drop, at the AC solution's weakest bus, medians over storms with a fold; beyond the fold Newton from the fine-tuned state must still "
        "converge nowhere",
        f"5. warning head for cases unresolved by the specified solver (baselines: GENCO's lowest voltage, residual and predicted-violation count alone; logistic regression on GENCO's own outputs and embeddings, trained on T1's training cases, threshold fixed on T1's "
        f"validation cases for {HEAD_RECALL:.0%} recall of unresolved cases): AUROC, recall and false alarms on F1, footprint and the K1 storms"],
    "secondary": ["Newton iterations from GENCO's state, verdict agreement with the reference, cases resolved only from the fine-tuned state "
                  "(checked with PYPOWER), the branch check on the fine-tuned workflow, end-to-end GENCO time (unchanged by construction)"],
    "bootstrap": BOOTSTRAP, "seed": SEED,
}
json.dump(T2_PROTOCOL, open(Path(OUT_DIR) / "frozen_protocol.json", "w"), indent=1, default=str)
print(json.dumps(T2_PROTOCOL["primary_endpoints"], indent=1))

In [ ]:
import zipfile, hashlib, glob
cand = [p_ for p_ in [T1_ZIP, f"/content/{T1_ZIP}"] + ([f"{DRIVE_DIR}/{T1_ZIP}"] if DRIVE_DIR else []) if os.path.exists(p_)]
if not cand and not os.path.exists(f"{T1_DIR}/genco_finetuned_state_dict.pt"):
    raise RuntimeError(f"{T1_ZIP} not found in MyDrive/genco_runs or the session: run finetune.ipynb first, or upload the zip and run this cell again")
if cand:
    with zipfile.ZipFile(cand[0]) as z:
        z.extractall(T1_DIR)
T1_SUMMARY = json.load(open(f"{T1_DIR}/summary.json"))
CK = f"{T1_DIR}/genco_finetuned_state_dict.pt"
CK_SHA = hashlib.sha256(open(CK, "rb").read()).hexdigest()
assert CK_SHA == T1_SUMMARY["checkpoint_sha256"], "the checkpoint does not match the one T1 recorded"
T1_CASES = pd.read_csv(f"{T1_DIR}/train_cases.csv")
T1_CASES["removed_lines"] = T1_CASES["removed_lines"].map(json.loads)
print(f"fine-tuned checkpoint {CK_SHA[:16]}... (selected at step {T1_SUMMARY['selected_step']} of {T1_SUMMARY['optimizer_steps']}); "
      f"{len(T1_CASES)} T1 cases ({int((T1_CASES.status == 'unresolved').sum())} unresolved)")

## 2. The cases, the download, the operating points

In [ ]:
import base64, io
dec = lambda s: pd.read_csv(io.StringIO(base64.b64decode(s).decode()))
F1C = dec(F1_CASES_B64); F1C["removed_lines"] = F1C["removed_lines"].map(json.loads)
K1C = dec(K1_CASES_B64); K1C["removed_lines"] = K1C["removed_lines"].map(json.loads)
K1S = dec(K1_SWEEP_B64)
rng_h = np.random.default_rng(SEED + 3)
train_bases = np.array(sorted(T1_CASES[T1_CASES.split == "train"].base.unique()))
HEAD_TRAIN_BASES = set(int(x) for x in rng_h.choice(train_bases, min(HEAD_TRAIN_BASES_N, len(train_bases)), replace=False))
HEAD_VAL_BASES = set(int(x) for x in T1_CASES[T1_CASES.split == "val"].base.unique())
HEADC = T1_CASES[T1_CASES.base.isin(HEAD_TRAIN_BASES | HEAD_VAL_BASES)].copy()
print(f"F1 {len(F1C)} cases on {F1C.base.nunique()} base points | K1 {len(K1C)} storms, {len(K1S)} sweep points | "
      f"head: {int(HEADC.base.isin(HEAD_TRAIN_BASES).sum())} training and {int(HEADC.base.isin(HEAD_VAL_BASES).sum())} validation cases")

In [ ]:
from huggingface_hub import HfApi, snapshot_download
all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True, revision=HF_REVISION) if getattr(f, "size", None) is not None]
plan = {p_: s_ for p_, s_ in all_files if p_.startswith(MODEL_DIR + "/")}
want = {LEVEL_BASE: set(N2_PARTS), **{lv: set(pp) for lv, pp in FORGET_PARTS.items() if lv != LEVEL_BASE}}
for p_, s_ in all_files:
    for lv, pp in want.items():
        m = RAW3_PAT.search(p_) if p_.startswith(f"data/test_{lv}/") else None
        if m and int(m.group(2)) in pp:
            plan[p_] = s_
new = sum(s_ for p_, s_ in plan.items() if not (Path(LOCAL_DIR) / p_).exists()) / 1e9
print(f"Planned: {len(plan)} files, {new:.2f} GB not yet on disk")
if new > MAX_DOWNLOAD_GB:
    raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB")
snapshot_download(REPO_ID, allow_patterns=list(plan), local_dir=LOCAL_DIR, revision=HF_REVISION)
print("Download finished")

In [ ]:
import pyarrow.dataset as ds_


def ids_by_part(level):
    t = ds_.dataset(str(find_local(LOCAL_DIR, level, "bus_data")), format="parquet", partitioning="hive").to_table(columns=["scenario", "scenario_partition"]).to_pandas().drop_duplicates()
    return {int(s): int(p_) for s, p_ in zip(t.scenario, t.scenario_partition)}


P2 = ids_by_part(LEVEL_BASE)
FORGET_IDS = {}
for lv, pp in FORGET_PARTS.items():
    part = P2 if lv == LEVEL_BASE else ids_by_part(lv)
    FORGET_IDS[lv] = np.array(sorted(s for s, p_ in part.items() if p_ in pp)[:N_FORGET], dtype=np.int64)
BASE_IDS = np.array(sorted(set(F1C.base) | set(K1C.base) | set(HEADC.base) | set(FORGET_IDS[LEVEL_BASE].tolist())), dtype=np.int64)
missing = [int(b) for b in BASE_IDS if b not in P2]
if missing:
    raise RuntimeError(f"{len(missing)} needed N-2 operating points are not on disk (first {missing[:5]})")
BASE = truth_dicts(LEVEL_BASE, BASE_IDS)
NB = next(iter(BASE.values()))["nb"]
TRUTH_L = {lv: truth_dicts(lv, ids) for lv, ids in FORGET_IDS.items() if lv != LEVEL_BASE}
TRUTH_L[LEVEL_BASE] = {int(s): BASE[int(s)] for s in FORGET_IDS[LEVEL_BASE]}
print(f"{len(BASE)} N-2 operating points loaded ({NB} buses); forgetting sets: " + ", ".join(f"N-{lv}: {len(v)}" for lv, v in TRUTH_L.items()))

## 3. The two models

In [ ]:
# [GPU] build graphs of the base points (used for the model check)

import yaml
cfg_path = os.path.join(GRAPHKIT_DIR, "scripts", "contingency", "configs", "caseTexas_eval.yaml")
with open(cfg_path) as f:
    ARGS = NestedNamespace(**yaml.safe_load(f))
stats = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "normalizer_stats.pt"), map_location="cpu", weights_only=True)
stats_net = stats[NETWORK] if NETWORK in stats else (stats if "baseMVA" in stats else next(iter(stats.values())))
NORMALIZER = load_normalizer(ARGS)
NORMALIZER.fit_from_dict(stats_net)
TRANSFORM = get_task_transforms(ARGS)
print("normalizer baseMVA:", NORMALIZER.baseMVA, "| config:", cfg_path)

BUS_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "bus_data"), BASE_IDS)
GEN_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "gen_data"), BASE_IDS)
BR_T = read_all(find_local(LOCAL_DIR, LEVEL_BASE, "branch_data"), BASE_IDS)
BUS_T = merge_agg_gen_into_bus(BUS_T, GEN_T)
BG, GG, RG = BUS_T.groupby("scenario"), GEN_T.groupby("scenario"), BR_T.groupby("scenario")


def build_graph(sid, base, Pd=None, Qd=None, removed_lines=()):
    """GENCO's input graph for base point `base` with extra lines removed and (optionally) other loads."""
    b = BG.get_group(base).sort_values("bus").reset_index(drop=True).copy()
    if Pd is not None:
        b["Pd"], b["Qd"] = np.asarray(Pd, float), np.asarray(Qd, float)
    r = RG.get_group(base).copy()
    if len(removed_lines):
        r.loc[r["idx"].astype(int).isin([int(x) for x in removed_lines]), "br_status"] = 0.0
    g = build_hetero_data_for_scenario(int(sid), b, GG.get_group(base), r)
    NORMALIZER.transform(g)
    return TRANSFORM(g)


def scaled_graph(g, lam):
    """The same graph with every load and every non-slack generator scaled by lam (the slack's generation is a
    masked input, 0, and stays 0; loads and generation are divided by baseMVA, so scaling the inputs is exact)."""
    h = g.clone()
    h["bus"].x = h["bus"].x.clone(); h["gen"].x = h["gen"].x.clone()
    h["bus"].x[:, PD_H] *= lam; h["bus"].x[:, QD_H] *= lam; h["gen"].x[:, PG_H] *= lam
    return h


print('graph builder ready')

In [ ]:
# GENCO's input graphs of released N-10 / N-20 scenarios (the level-2 builder of the cell above covers N-2)
LEVEL_TABLES = {}
for lv, ids in FORGET_IDS.items():
    if lv == LEVEL_BASE:
        continue
    bt = read_all(find_local(LOCAL_DIR, lv, "bus_data"), ids); gt = read_all(find_local(LOCAL_DIR, lv, "gen_data"), ids)
    rt = read_all(find_local(LOCAL_DIR, lv, "branch_data"), ids)
    bt = merge_agg_gen_into_bus(bt, gt)
    LEVEL_TABLES[lv] = (bt.groupby("scenario"), gt.groupby("scenario"), rt.groupby("scenario"))


def level_graph(lv, s):
    if lv == LEVEL_BASE:
        return build_graph(int(s), int(s))
    bg, gg, rg = LEVEL_TABLES[lv]
    b = bg.get_group(int(s)).sort_values("bus").reset_index(drop=True).copy()
    g = build_hetero_data_for_scenario(int(s), b, gg.get_group(int(s)), rg.get_group(int(s)).copy())
    NORMALIZER.transform(g)
    return TRANSFORM(g)

In [ ]:
# [GPU] load model

TASK = get_task(ARGS, [NORMALIZER])
sd = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "best_model_state_dict.pt"), map_location="cpu")
sd = adapt_state_dict_for_model(sd, TASK)
missing_keys, unexpected_keys = TASK.load_state_dict(sd, strict=False)
print(f"checkpoint loaded: {len(sd)} tensors | missing {len(missing_keys)} | unexpected {len(unexpected_keys)}")
if missing_keys:
    raise RuntimeError(f"The checkpoint does not match the model. Missing: {missing_keys[:5]}")
MODEL = TASK.model.to(DEVICE).eval()
print("GENCO parameters:", sum(p.numel() for p in MODEL.parameters()))


def forward(model, batch):
    return model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict, batch.mask_dict)

In [ ]:
import copy
MODEL0 = MODEL
TASK1 = get_task(ARGS, [NORMALIZER])
sd1 = torch.load(CK, map_location="cpu")
TASK1.model.load_state_dict(sd1, strict=True)
MODEL1 = TASK1.model.to(DEVICE).eval()
MODELS = {"released": MODEL0, "fine_tuned": MODEL1}
n_diff = sum(int(not torch.equal(a.cpu(), b.cpu())) for a, b in zip(MODEL0.state_dict().values(), MODEL1.state_dict().values()))
print(f"two models loaded; {n_diff} of {len(sd1)} tensors differ between them")


def run_model(model, graphs, bs=INFER_BATCH):
    """(vm, va in degrees, embedding mean, embedding max) per graph, in order; the embedding is the input of the bus read-out layer."""
    rec = []
    h = model.mlp_bus.register_forward_pre_hook(lambda m, i: rec.append(i[0].detach()))
    out = [None] * len(graphs)
    try:
        for i, g in enumerate(graphs):
            g["scenario_id"] = torch.tensor([i], dtype=torch.long)
        for b in DataLoader(graphs, batch_size=bs, shuffle=False):
            rec.clear()
            b = b.to(DEVICE)
            with torch.no_grad():
                o = forward(model, b)["bus"].float()
            hid = rec[-1].float()
            ptr = b["bus"].ptr.cpu().tolist()
            for j, i in enumerate(b["scenario_id"].view(-1).cpu().tolist()):
                lo, hi = ptr[j], ptr[j + 1]
                out[int(i)] = (o[lo:hi, VM_OUT].cpu().numpy().astype(np.float64), np.degrees(o[lo:hi, VA_OUT].cpu().numpy().astype(np.float64)),
                               hid[lo:hi].mean(0).cpu().numpy(), hid[lo:hi].max(0).values.cpu().numpy())
    finally:
        h.remove()
    return out


chk = list(FORGET_IDS[LEVEL_BASE][:N_CHECK])
for name, m in MODELS.items():
    o = run_model(m, [build_graph(int(s), int(s)) for s in chk])
    e = [np.max(np.abs(vm - BASE[int(s)]["Vm"])[BASE[int(s)]["PQ"]]) for s, (vm, va, _, _) in zip(chk, o)]
    print(f"{name}: worst-bus voltage error on {len(chk)} released N-2 scenarios: median {np.median(e):.4f} p.u.")

## 4. Clustered cases: frozen-test and the held-out footprint construction (endpoints 1 and 2)

In [ ]:
# The contingency constructions of frozen-test / construction (uniform, nearest lines, corridor, soft footprint), unchanged
import networkx as nx
d0 = next(iter(BASE.values()))
G = nx.Graph()
for f_, t_ in zip(d0["f"][d0["on"]], d0["t"][d0["on"]]):
    G.add_edge(int(f_), int(t_))
t0 = time.perf_counter()
EB = nx.edge_betweenness_centrality(G, k=min(BETWEENNESS_SAMPLES, G.number_of_nodes()), seed=SEED, normalized=True)
LINE_BTW = np.array([EB.get((int(f_), int(t_)), EB.get((int(t_), int(f_)), 0.0)) for f_, t_ in zip(d0["f"], d0["t"])])
print(f"edge betweenness of the grid in {time.perf_counter() - t0:.0f} s")


def pick_lines(d, extra, mode, rng):
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    centre = int(rng.integers(nb))
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    hop = dijkstra(A, directed=False, unweighted=True, indices=centre)
    dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
    if mode == "nearest":
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    elif mode == "uniform":
        order = rng.permutation(cand)
    elif mode == "corridor":
        h = REGION_HOPS
        while (dist <= h).sum() < 3 * extra and h < 12:
            h += 1
        reg = cand[dist <= h]
        order = reg[np.lexsort((rng.random(len(reg)), -LINE_BTW[reg]))]
    elif mode == "footprint":
        sel = dist <= FOOT_HOPS
        reg, w = cand[sel], np.exp(-dist[sel] / FOOT_SCALE)
        keys = rng.random(len(reg)) ** (1.0 / w)
        order = reg[np.argsort(-keys)]
    else:
        raise ValueError(mode)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, np.nan, centre
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if removed else 0.0
    return np.array(removed, dtype=int), radius, centre

In [ ]:
def rows_of(d, removed):
    return np.nonzero(np.isin(d["line"], np.asarray(removed, dtype=np.int64)))[0]


def case_dict(d, rows, lam=1.0):
    d2 = {kk: (v.copy() if isinstance(v, np.ndarray) else v) for kk, v in d.items()}
    if len(rows):
        d2["on"][rows] = False
    d2.pop("Va_dc", None)
    if lam != 1.0:
        d2["Pd"] = d2["Pd"] * lam; d2["Qd"] = d2["Qd"] * lam; d2["Pg"] = np.where(d2["REF"], d2["Pg"], d2["Pg"] * lam)
    return d2


def decisions_of(ver, V):
    lim = ver.classify(V)
    ld = ver.loading(V)
    return {"over": set(lim["overloaded_lines"]), "flag_m": set(ver.line_idx[ld >= POLICY["line_limit"] - LINE_MARGIN].astype(int).tolist()),
            "low": set(lim["low_voltage_buses"]), "high": set(lim["high_voltage_buses"]), "vm": np.abs(V)}


def genco_vs_ref(ver, vm, va_deg, ref):
    """What GENCO alone decides against a reference state's decisions (both from the same Verifier, so the same limits)."""
    g = decisions_of(ver, ver.state(vm, va_deg))
    pq = ver.d["PQ"]
    mo = len(ref["over"] - g["over"]); mom = len(ref["over"] - g["flag_m"])
    mv = len(ref["low"] - g["low"]) + len(ref["high"] - g["high"])
    return {"worst_err": float(np.max(np.abs(g["vm"] - ref["vm"])[pq])), "missed_over": mo, "missed_over_m": mom, "missed_v": mv,
            "missed": mo + mv, "missed_m": mom + mv, "any_miss": int(mo + mv > 0), "any_miss_m": int(mom + mv > 0),
            "ref_decisions": len(ref["over"]) + len(ref["low"]) + len(ref["high"]), "genco_min_vm": float(g["vm"][pq].min())}


def evaluate_cases(cases, label):
    """cases: list of dicts (sid, base, family, rows, lam). Reference: the classical workflow; GENCO of both models; the GENCO-first
    workflow with each model's state; GENCO-alone decisions against the reference."""
    t0 = time.perf_counter()
    graphs = {}
    outs = {}
    for name, m in MODELS.items():
        gl = []
        for c in cases:
            g = build_graph(c["sid"], c["base"], removed_lines=c["removed_lines"])
            gl.append(scaled_graph(g, c["lam"]) if c["lam"] != 1.0 else g)
        outs[name] = run_model(m, gl)
    rows = []
    for i, c in enumerate(cases):
        d2 = case_dict(BASE[c["base"]], c["rows"], c["lam"])
        ver = Verifier(d2, POLICY)
        ref = ver.run(policy_name="classical")
        row = {"sid": c["sid"], "base": c["base"], "family": c["family"], "set": label, "ref_status": ref.status, "ref_start": ref.start_used}
        refd = None
        if ref.converged:
            refd = decisions_of(ver, np.array(ref.state["vm"]) * np.exp(1j * np.radians(np.array(ref.state["va_deg"]))))
        for name in MODELS:
            vm, va, em, ex = outs[name][i]
            r = ver.run(genco_vm=vm, genco_va_deg=va, policy_name="genco_verify")
            row.update({f"{name}_status": r.status, f"{name}_start": r.start_used, f"{name}_iters_from_genco": next((h["iterations"] for h in r.history if h["start"] == "genco" and h["converged"]), np.nan),
                        f"{name}_genco_converged": any(h["start"] == "genco" and h["converged"] for h in r.history),
                        f"{name}_resP_p99_MW": r.genco["resP_p99_MW"], f"{name}_resP_max_MW": r.genco["resP_max_MW"], f"{name}_mismatch_pu": r.genco["mismatch_pu"],
                        f"{name}_min_vm": r.genco["min_vm"], f"{name}_n_low_v": r.genco["n_low_v"], f"{name}_n_over_m": len(r.genco["flagged_lines_margin"]),
                        f"{name}_max_loading": r.genco["max_loading"]})
            if refd is not None:
                row.update({f"{name}_{k}": v for k, v in genco_vs_ref(ver, vm, va, refd).items()})
                if r.converged:
                    row[f"{name}_same_verdict"] = (r.status == ref.status) and (np.abs(np.array(r.state["vm"]) - refd["vm"]).max() < 1e-4)
            FEAT[(label, c["sid"], name)] = np.r_[[r.genco["min_vm"], r.genco["resP_p99_MW"], r.genco["resP_max_MW"], np.log10(r.genco["mismatch_pu"] + 1e-12),
                                                   r.genco["n_low_v"], len(r.genco["flagged_lines_margin"]), r.genco["max_loading"],
                                                   float(np.mean(np.asarray(r.genco["vm"])[ver.d["PQ"]])), float(np.percentile(np.asarray(r.genco["vm"])[ver.d["PQ"]], 1))], em, ex]
        rows.append(row)
    print(f"{label}: {len(cases)} cases in {time.perf_counter() - t0:.0f} s")
    return pd.DataFrame(rows)


def cluster_boot(df, col0, col1, cluster="base", B=None, seed=SEED):
    """Mean of the paired difference col1 - col0 with a 95 % bootstrap interval over clusters."""
    B = B or BOOTSTRAP
    x = df[[cluster, col0, col1]].dropna()
    diff = (x[col1] - x[col0]).to_numpy(); cl = x[cluster].to_numpy()
    keys, inv = np.unique(cl, return_inverse=True)
    s = np.bincount(inv, weights=diff); n = np.bincount(inv)
    rng = np.random.default_rng(seed)
    bs = []
    for _ in range(B):
        k = rng.integers(0, len(keys), len(keys))
        bs.append(s[k].sum() / n[k].sum())
    return float(diff.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5)), int(len(diff))


def mcnemar(df, col0, col1):
    """Exact two-sided McNemar test on paired binary outcomes: (cases worse only with col0, only with col1, p-value)."""
    from scipy.stats import binomtest
    x = df[[col0, col1]].dropna().astype(int)
    b = int(((x[col0] == 1) & (x[col1] == 0)).sum()); c = int(((x[col0] == 0) & (x[col1] == 1)).sum())
    p = binomtest(min(b, c), b + c, 0.5).pvalue if b + c > 0 else 1.0
    return b, c, float(p)


FEAT = {}

In [ ]:
F1_LIST = []
for r_ in F1C.itertuples(index=False):
    d = BASE[int(r_.base)]
    F1_LIST.append({"sid": int(r_.sid), "base": int(r_.base), "family": r_.family, "removed_lines": list(r_.removed_lines),
                    "rows": rows_of(d, r_.removed_lines), "lam": 1.0})
FP_LIST, skipped = [], 0
for b_id in sorted(F1C.base.unique()):
    d = BASE[int(b_id)]
    rng = np.random.default_rng(SEED * 1000003 + int(b_id) + 11)
    extra = 20 - int((~d["on"]).sum())
    rem, radius, centre = pick_lines(d, extra, "footprint", rng)
    if rem is None:
        skipped += 1; continue
    FP_LIST.append({"sid": int(b_id) * 10 + 9, "base": int(b_id), "family": "footprint-N20", "removed_lines": d["line"][rem].astype(int).tolist(),
                    "rows": rem, "lam": 1.0})
print(f"F1: {len(F1_LIST)} cases; footprint (held out of fine-tuning): {len(FP_LIST)} cases ({skipped} could not be built)")
pd.DataFrame([{k: c[k] for k in ("sid", "base", "family")} | {"removed_lines": json.dumps(c["removed_lines"])} for c in FP_LIST]).to_csv(
    Path(OUT_DIR) / "footprint_cases.csv", index=False)
EV_F1 = evaluate_cases(F1_LIST, "F1")
EV_FP = evaluate_cases(FP_LIST, "footprint")
EV = pd.concat([EV_F1, EV_FP], ignore_index=True)
EV.to_csv(Path(OUT_DIR) / "cases.csv", index=False)
print(f"reference verdicts (classical): {EV.groupby('set').ref_status.value_counts().to_dict()}")

In [ ]:
verified = EV[EV.ref_status != "unresolved"].copy()
verified["group"] = np.where(verified.set == "footprint", "footprint (held out)",
                             np.where(verified.family.str.startswith("uniform"), "uniform (F1)", "clustered (F1)"))
E12 = []
for grp, g in verified.groupby("group"):
    for metric in ["missed", "missed_m", "worst_err"]:
        m, lo, hi, n = cluster_boot(g, f"released_{metric}", f"fine_tuned_{metric}")
        E12.append({"group": grp, "metric": metric, "cases": n, "released_mean": float(g[f"released_{metric}"].mean()), "fine_tuned_mean": float(g[f"fine_tuned_{metric}"].mean()),
                    "released_median": float(g[f"released_{metric}"].median()), "fine_tuned_median": float(g[f"fine_tuned_{metric}"].median()),
                    "diff_mean": m, "diff_ci_lo": lo, "diff_ci_hi": hi})
    for metric in ["any_miss", "any_miss_m"]:
        b, c, p = mcnemar(g, f"released_{metric}", f"fine_tuned_{metric}")
        m, lo, hi, n = cluster_boot(g, f"released_{metric}", f"fine_tuned_{metric}")
        E12.append({"group": grp, "metric": metric, "cases": len(g), "released_mean": float(g[f"released_{metric}"].mean()), "fine_tuned_mean": float(g[f"fine_tuned_{metric}"].mean()),
                    "diff_mean": m, "diff_ci_lo": lo, "diff_ci_hi": hi,       # the primary analysis: paired cluster bootstrap over base points
                    "fixed_by_fine_tuning": b, "broken_by_fine_tuning": c,
                    "mcnemar_p_ignores_clustering": p})
E12 = pd.DataFrame(E12); E12.to_csv(Path(OUT_DIR) / "endpoints_cases.csv", index=False)
display(E12)
FAMT = verified.groupby("family")[["released_missed", "fine_tuned_missed", "released_missed_m", "fine_tuned_missed_m", "released_worst_err", "fine_tuned_worst_err",
                                   "released_ref_decisions"]].agg(["mean", "median"]).round(4)
display(FAMT)
SEC = {}
for name in MODELS:
    v_ = EV[EV.ref_status != "unresolved"]
    SEC[name] = {"iters_from_genco_mean": float(v_[f"{name}_iters_from_genco"].mean()), "genco_start_converged": int(v_[f"{name}_genco_converged"].sum()),
                 "verified_cases": int(len(v_)), "same_verdict_and_state": int(v_[f"{name}_same_verdict"].eq(True).sum()) if f"{name}_same_verdict" in v_ else 0,
                 "resolved_only_with_this_genco": int(((EV.ref_status == "unresolved") & (EV[f"{name}_status"] != "unresolved")).sum())}
print("secondary:", json.dumps(SEC, indent=1))

In [ ]:
only = EV[(EV.ref_status == "unresolved") & (EV.fine_tuned_status != "unresolved")]
PCHK = []
if len(only):
    from pypower.api import ppoption
    preload_raw(LOCAL_DIR, LEVEL_BASE, sorted(only.base.unique()))
    for r_ in only.itertuples(index=False):
        c = next(x for x in F1_LIST + FP_LIST if x["sid"] == r_.sid)
        d2 = case_dict(BASE[c["base"]], c["rows"])
        g = build_graph(c["sid"], c["base"], removed_lines=c["removed_lines"])
        vm, va, _, _ = run_model(MODEL1, [g])[0]
        rr = Verifier(d2, dict(POLICY, branch_check=True)).run(genco_vm=vm, genco_va_deg=va, policy_name="genco_verify")
        ind = Indep(*raw_tables(LOCAL_DIR, LEVEL_BASE, c["base"]))
        Vm_, Va_ = np.array(rr.state["vm"]), np.array(rr.state["va_deg"])
        x = ind.solve(removed_idx=c["removed_lines"], lam=1.0, start=(Vm_, Va_), max_it=10)
        PCHK.append({"sid": c["sid"], "family": c["family"], "min_vm_all": float(Vm_.min()), "status": rr.status,
                     "pypower_mismatch_pu": ind.mismatch(ind.ppc(c["removed_lines"], 1.0), Vm_ * np.exp(1j * np.radians(Va_))),
                     "pypower_converged_from_state": x["converged"], "pypower_max_dVm": float(np.abs(x["Vm"] - Vm_).max()),
                     "pypower_classical_any": any(ind.solve(removed_idx=c["removed_lines"], lam=1.0, start=s_, max_it=100)["converged"] for s_ in ["base", "flat", "dc"]),
                     "n_solutions_found": rr.flags.get("n_solutions_found"), "low_voltage_flag": rr.flags.get("low_voltage_branch_suspect")})
    PCHK = pd.DataFrame(PCHK); PCHK.to_csv(Path(OUT_DIR) / "resolved_only_finetuned.csv", index=False); display(PCHK)
else:
    print("no case is resolved only from the fine-tuned state")

## 5. Forgetting on released scenarios (endpoint 3)

In [ ]:
FG = []
for lv, ids in FORGET_IDS.items():
    graphs = [level_graph(lv, s) for s in ids]
    outs = {name: run_model(m, graphs) for name, m in MODELS.items()}
    for i, s in enumerate(ids):
        d = TRUTH_L[lv][int(s)]
        ver = Verifier(case_dict(d, []), POLICY)
        ref = decisions_of(ver, ver.state(d["Vm"], d["Va"]))
        row = {"level": lv, "scenario": int(s)}
        for name in MODELS:
            vm, va, _, _ = outs[name][i]
            row.update({f"{name}_{k}": v for k, v in genco_vs_ref(ver, vm, va, ref).items()})
        FG.append(row)
FG = pd.DataFrame(FG); FG.to_csv(Path(OUT_DIR) / "forgetting.csv", index=False)
E3 = []
for lv, g in FG.groupby("level"):
    m, lo, hi, n = cluster_boot(g, "released_worst_err", "fine_tuned_worst_err", cluster="scenario")
    mm, mlo, mhi, _ = cluster_boot(g, "released_missed_m", "fine_tuned_missed_m", cluster="scenario")
    E3.append({"level": f"N-{lv}", "scenarios": n, "released_median_err": float(g.released_worst_err.median()), "fine_tuned_median_err": float(g.fine_tuned_worst_err.median()),
               "diff_mean_err": m, "diff_ci_lo": lo, "diff_ci_hi": hi, "non_inferior": bool(hi < NONINF_MARGIN_PU),
               "released_missed_m_per_case": float(g.released_missed_m.mean()), "fine_tuned_missed_m_per_case": float(g.fine_tuned_missed_m.mean()),
               "diff_missed_m": mm, "diff_missed_m_ci": [mlo, mhi]})
E3 = pd.DataFrame(E3); E3.to_csv(Path(OUT_DIR) / "endpoint_forgetting.csv", index=False); display(E3)

## 6. Near the fold: the storms (endpoint 4)

In [ ]:
t0 = time.perf_counter()
KST = []
for r_ in K1C.itertuples(index=False):
    d = BASE[int(r_.base)]
    rows = rows_of(d, r_.removed_lines)
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][rows] = False
    d2["Va_dc"] = dc_angles(d2)
    case = LambdaCase(d2)
    V1, who = case.solve_any(1.0, [("base", case.state(d["Vm"], d["Va"])), ("flat", case.flat), ("dc", case.dc)], 30, TOL)
    res = find_lambda_max(case, V1, step=LAM_STEP, width=LAM_WIDTH, lam_min=LAM_MIN, lam_cap=LAM_CAP, tol=TOL)
    has_nose = bool(np.isfinite(res["lam_max"]) and not res["capped"])
    lams = sorted(K1S[K1S.sid == r_.sid].lam.tolist())
    truth = {}
    if has_nose:
        truth = truth_below_nose(case, res["lam_max"], res["V_max"], [l for l in lams if l < res["lam_max"] - 1e-9], tol=TOL)
        truth[res["lam_max"]] = res["V_max"]
    if V1 is not None:
        truth[1.0] = V1
    KST.append({"sid": int(r_.sid), "base": int(r_.base), "removed_lines": list(r_.removed_lines), "case": case, "solvable_at_1": V1 is not None,
                "lam_max": res["lam_max"], "lam_max_K1": float(r_.lam_max), "has_nose": has_nose, "lams": lams, "truth": truth, "Pd": d["Pd"], "Qd": d["Qd"]})
print(f"{len(KST)} storms rebuilt, folds recomputed in {time.perf_counter() - t0:.0f} s; largest difference to K1's fold: "
      f"{np.nanmax([abs(s['lam_max'] - s['lam_max_K1']) for s in KST]):.2e}")
KOUT = {}
for name, m in MODELS.items():
    gl, keys = [], []
    for s in KST:
        g0 = build_graph(s["sid"], s["base"], s["Pd"], s["Qd"], s["removed_lines"])
        for lam in s["lams"]:
            gl.append(scaled_graph(g0, lam)); keys.append((s["sid"], lam))
    o = run_model(m, gl)
    KOUT[name] = {k: (vm, va) for k, (vm, va, _, _) in zip(keys, o)}
    for k, (vm, va, em, ex) in zip(keys, o):
        FEAT[("K1", k, name)] = (vm, va, em, ex)
rows = []
for s in KST:
    case = s["case"]
    lmax = s["lam_max"] if s["has_nose"] else np.nan
    weak = int(case.decisions(lmax, s["truth"][lmax])["weak_bus"]) if s["has_nose"] else -1
    for lam in s["lams"]:
        Vt = s["truth"].get(lam)
        row = {"sid": s["sid"], "solvable_at_1": s["solvable_at_1"], "has_nose": s["has_nose"], "lam": lam, "lam_max": lmax,
               "lam_rel": lam - lmax if s["has_nose"] else np.nan, "beyond_fold": bool(s["has_nose"] and lam > lmax + 1e-9), "at_actual_load": abs(lam - 1) < 1e-9,
               "truth_exists": Vt is not None}
        if Vt is not None:
            dt = case.decisions(lam, Vt); ldt = case.loading(Vt)
            row.update({"true_min_vm": dt["min_vm"], "true_vm_weak": float(np.abs(Vt[weak])) if weak >= 0 else np.nan, "true_n_over": int((ldt >= 1).sum()), "true_n_low_v": dt["n_low_v"]})
        for name in MODELS:
            vm, va = KOUT[name][(s["sid"], lam)]
            Vg = case.state(vm, va)
            dg = case.decisions(lam, Vg); ldg = case.loading(Vg)
            row.update({f"{name}_min_vm": dg["min_vm"], f"{name}_vm_weak": float(np.abs(Vg[weak])) if weak >= 0 else np.nan, f"{name}_n_low_v": dg["n_low_v"],
                        f"{name}_n_over_m": int((ldg >= 1 - LINE_MARGIN).sum()),
                        f"{name}_flags": bool(dg["n_low_v"] > 0 or dg["n_high_v"] > 0 or (ldg >= 1 - LINE_MARGIN).any())})
            if Vt is not None:
                tov = ldt >= 1
                row[f"{name}_missed_over_m_share"] = float((tov & ~(ldg >= 1 - LINE_MARGIN)).sum() / tov.sum()) if tov.sum() else np.nan
            if name == "fine_tuned" and row["beyond_fold"]:
                _, okk, _, _ = case.solve(lam, Vg, 30, TOL)
                row["fine_tuned_newton_converged_beyond"] = bool(okk)
        rows.append(row)
KS = pd.DataFrame(rows); KS.to_csv(Path(OUT_DIR) / "near_fold.csv", index=False)
chk_ = KS.merge(K1S[["sid", "lam", "genco_min_vm"]], on=["sid", "lam"], how="inner")
print(f"the released model reproduces K1's GENCO lowest voltage to {np.nanmax(np.abs(chk_.released_min_vm - chk_.genco_min_vm)):.1e} p.u. (median {np.nanmedian(np.abs(chk_.released_min_vm - chk_.genco_min_vm)):.1e})")


def fold_summary(sub):
    out = {"storms": int(sub.sid.nunique())}
    at = sub[np.isclose(sub.lam_rel, 0.0, atol=1e-6)]
    pre = sub[np.isclose(sub.lam_rel, -0.05, atol=1e-6)].set_index("sid")
    atf = at.set_index("sid")
    common = atf.index.intersection(pre.index)
    out["lowest_v_at_fold"] = {k: float(at[f"{k}_min_vm"].median()) for k in ["true"] + list(MODELS)}
    for k in ["true"] + list(MODELS):
        col = f"{k}_vm_weak"
        out.setdefault("drop_last_5pct", {})[k] = float((pre.loc[common, col] - atf.loc[common, col]).median())
    tdrop = (pre.loc[common, "true_vm_weak"] - atf.loc[common, "true_vm_weak"]).to_numpy()
    for k in MODELS:
        gdrop = (pre.loc[common, f"{k}_vm_weak"] - atf.loc[common, f"{k}_vm_weak"]).to_numpy()
        out.setdefault("drop_share_of_ac", {})[k] = float(np.median(gdrop / np.where(np.abs(tdrop) > 1e-9, tdrop, np.nan)))
        out.setdefault("missed_overloads_at_fold_m", {})[k] = float(at[f"{k}_missed_over_m_share"].median())
    beyond = sub[sub.beyond_fold]
    out["beyond_fold_points"] = int(len(beyond))
    out["fine_tuned_newton_converged_beyond"] = int(beyond["fine_tuned_newton_converged_beyond"].eq(True).sum()) if "fine_tuned_newton_converged_beyond" in beyond else 0
    out["calm_beyond_fold"] = {k: int((~beyond.groupby("sid")[f"{k}_flags"].any()).sum()) for k in MODELS}
    return out


E4 = {"unresolved_at_actual_load": fold_summary(KS[KS.has_nose & ~KS.solvable_at_1]), "solved_at_actual_load": fold_summary(KS[KS.has_nose & KS.solvable_at_1])}
a1 = KS[KS.at_actual_load & ~KS.solvable_at_1]
E4["unresolved_storms_at_actual_load"] = {k: {"min_vm_median": float(a1[f"{k}_min_vm"].median()), "n_low_v_median": float(a1[f"{k}_n_low_v"].median()),
                                              "n_over_m_median": float(a1[f"{k}_n_over_m"].median())} for k in MODELS}
json.dump(E4, open(Path(OUT_DIR) / "endpoint_near_fold.json", "w"), indent=1, default=float)
print(json.dumps(E4, indent=1, default=float))

## 7. The warning head for unresolved cases (endpoint 5)

In [ ]:
# Features: what GENCO itself outputs (lowest and mean voltage, residuals, flagged lines and buses) and its bus embeddings; no solver
t0 = time.perf_counter()
HL = []
for name, m in MODELS.items():
    gl = []
    for r_ in HEADC.itertuples(index=False):
        g = build_graph(int(r_.cid), int(r_.base), removed_lines=r_.removed_lines)
        gl.append(scaled_graph(g, float(r_.lam)) if float(r_.lam) != 1.0 else g)
    o = run_model(m, gl)
    for (r_, (vm, va, em, ex)) in zip(HEADC.itertuples(index=False), o):
        d = BASE[int(r_.base)]
        ver = Verifier(case_dict(d, rows_of(d, r_.removed_lines), float(r_.lam)), POLICY)
        Vg = ver.state(vm, va); lim = ver.classify(Vg)
        mm = (Vg * np.conj(ver.Y @ Vg) - ver.S) * SN_MVA; rp = np.abs(mm.real[ver.nonref])
        ld = ver.loading(Vg)
        FEAT[("head", int(r_.cid), name)] = np.r_[[lim["min_vm"], np.percentile(rp, 99), rp.max(), np.log10(ver.mismatch(Vg) + 1e-12), lim["n_low_v"],
                                                   int((ld >= 1 - LINE_MARGIN).sum()), lim["max_loading"], float(np.mean(np.abs(Vg)[d["PQ"]])),
                                                   float(np.percentile(np.abs(Vg)[d["PQ"]], 1))], em, ex]
print(f"head features for {len(HEADC)} T1 cases x 2 models in {time.perf_counter() - t0:.0f} s")

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
y_tr = (HEADC[HEADC.base.isin(HEAD_TRAIN_BASES)].status == "unresolved").to_numpy().astype(int)
y_va = (HEADC[HEADC.base.isin(HEAD_VAL_BASES)].status == "unresolved").to_numpy().astype(int)
ids_tr = HEADC[HEADC.base.isin(HEAD_TRAIN_BASES)].cid.astype(int).tolist(); ids_va = HEADC[HEADC.base.isin(HEAD_VAL_BASES)].cid.astype(int).tolist()


def k1_feat(name, key):
    vm, va, em, ex = FEAT[("K1", key, name)]
    s = next(x for x in KST if x["sid"] == key[0]); case = s["case"]
    Vg = case.state(vm, va); dg = case.decisions(key[1], Vg); ld = case.loading(Vg)
    pq = case.PQ
    return np.r_[[dg["min_vm"], dg["resP_p99_MW"], dg["resP_max_MW"], np.log10(dg["mismatch_pu"] + 1e-12), dg["n_low_v"], int((ld >= 1 - LINE_MARGIN).sum()),
                  dg["max_loading"], float(np.mean(np.abs(Vg)[pq])), float(np.percentile(np.abs(Vg)[pq], 1))], em, ex]


HEAD = {}
E5 = []
SCORES = []
FEATS_OUT = {}
for name in MODELS:
    X_tr = np.stack([FEAT[("head", c, name)] for c in ids_tr]); X_va = np.stack([FEAT[("head", c, name)] for c in ids_va])
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=5000, class_weight="balanced")).fit(X_tr, y_tr)
    p_va = clf.predict_proba(X_va)[:, 1]
    thr = float(np.quantile(p_va[y_va == 1], 1 - HEAD_RECALL)) if y_va.sum() else 0.5
    HEAD[name] = (clf, thr)
    tests = {}
    for lab, df in [("F1", EV[EV.set == "F1"]), ("footprint", EV[EV.set == "footprint"])]:
        if not len(df):
            continue
        X = np.stack([FEAT[(lab, int(s), name)] for s in df.sid]); y = (df.ref_status == "unresolved").to_numpy().astype(int)
        tests[lab] = (X, y, df.base.to_numpy(), [f"{lab}:{int(s)}" for s in df.sid])
    ka = KS[KS.at_actual_load]
    tests["K1 storms at the actual load"] = (np.stack([k1_feat(name, (int(r_.sid), float(r_.lam))) for r_ in ka.itertuples(index=False)]),
                                             (~ka.solvable_at_1).to_numpy().astype(int), ka.sid.to_numpy(), [f"K1:{int(r_.sid)}@{float(r_.lam):.4f}" for r_ in ka.itertuples(index=False)])
    kn = KS[KS.has_nose & (KS.lam_rel.abs() <= 0.05 + 1e-9)]
    tests["K1 points within 5 % of the fold"] = (np.stack([k1_feat(name, (int(r_.sid), float(r_.lam))) for r_ in kn.itertuples(index=False)]),
                                                 kn.beyond_fold.to_numpy().astype(int), kn.sid.to_numpy(), [f"K1:{int(r_.sid)}@{float(r_.lam):.4f}" for r_ in kn.itertuples(index=False)])
    tests["head training (T1)"] = (X_tr, y_tr, HEADC[HEADC.base.isin(HEAD_TRAIN_BASES)].base.to_numpy(), [f"T1:{c}" for c in ids_tr])
    tests["head validation (T1)"] = (X_va, y_va, HEADC[HEADC.base.isin(HEAD_VAL_BASES)].base.to_numpy(), [f"T1:{c}" for c in ids_va])
    for lab, (X, y, cl, ids) in tests.items():
        p = clf.predict_proba(X)[:, 1]
        SCORES.extend({"model": name, "test": lab, "case": i_, "label_unresolved": int(y_), "score": float(p_), "flagged": bool(p_ >= thr),
                       "genco_lowest_voltage": float(x_[0]), "genco_resP_p99_MW": float(x_[1])} for i_, y_, p_, x_ in zip(ids, y, p, X))
        FEATS_OUT[(name, lab)] = (X, y, ids)
        auc = float(roc_auc_score(y, p)) if 0 < y.sum() < len(y) else np.nan
        rng = np.random.default_rng(SEED + 7); keys = np.unique(cl); aucs = []
        for _ in range(min(BOOTSTRAP, 500)):
            kk = rng.choice(keys, len(keys)); idx = np.concatenate([np.nonzero(cl == k_)[0] for k_ in kk])
            if 0 < y[idx].sum() < len(idx):
                aucs.append(roc_auc_score(y[idx], p[idx]))
        flag = p >= thr
        base_auc = float(roc_auc_score(y, -X[:, 0])) if 0 < y.sum() < len(y) else np.nan          # GENCO's own lowest voltage, inverted
        res_auc = float(roc_auc_score(y, X[:, 1])) if 0 < y.sum() < len(y) else np.nan            # GENCO's own residual
        sev_auc = float(roc_auc_score(y, X[:, 4] + X[:, 5] - X[:, 0])) if 0 < y.sum() < len(y) else np.nan   # GENCO's predicted violations (low-voltage buses + flagged lines)
        E5.append({"model": name, "test": lab, "cases": int(len(y)), "unresolved": int(y.sum()), "auroc": auc,
                   "auroc_ci": [float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))] if aucs else None,
                   "recall_at_threshold": float(flag[y == 1].mean()) if y.sum() else np.nan, "false_alarm_rate": float(flag[y == 0].mean()) if (y == 0).any() else np.nan,
                   "flagged": int(flag.sum()), "auroc_genco_lowest_voltage_alone": base_auc, "auroc_genco_residual_alone": res_auc,
                   "auroc_genco_predicted_violations_alone": sev_auc, "threshold": thr})
E5 = pd.DataFrame(E5); E5.to_csv(Path(OUT_DIR) / "endpoint_head.csv", index=False); display(E5)
# ---- the warning, exported so that it can be reused and checked without re-running anything
import joblib, sklearn
HD = Path(OUT_DIR) / "unresolved_warning"; HD.mkdir(exist_ok=True)
SCALAR_FEATURES = ["lowest PQ-bus voltage of GENCO's state (p.u.)", "99th percentile of |P mismatch| at non-slack buses (MW)", "largest |P mismatch| (MW)",
                   "log10 of the largest power mismatch (p.u.)", "number of PQ buses outside their voltage limits", "number of lines at or above (1 - 0.0454) of their rating",
                   "largest line loading (share of rating)", "mean PQ-bus voltage (p.u.)", "1st percentile of PQ-bus voltages (p.u.)"]
hid = (FEATS_OUT[("released", "F1")][0].shape[1] - len(SCALAR_FEATURES)) // 2
FEATURE_DEFS = {"order": SCALAR_FEATURES + [f"bus embedding, mean over buses, dim {i}" for i in range(hid)] + [f"bus embedding, max over buses, dim {i}" for i in range(hid)],
                "embedding": "the input of GENCO's bus read-out layer (mlp_bus) after the last of its 12 layers, for the GENCO model named in the file",
                "state": "GENCO's predicted Vm (PQ buses) and Va, with PV and slack magnitudes at their set-points and the slack angle fixed (genco_verify.Verifier.state)",
                "label": "1 = unresolved by the specified solver: no start of genco_verify's classical policy (pre-contingency, DC, flat; lightsim2grid KLU, "
                         "tolerance 1e-8 p.u., 30 iterations) converged. A label of non-convergence of that solver, not a proof that no AC solution exists",
                "threshold_rule": f"the {1 - HEAD_RECALL:.0%} quantile of the scores of unresolved T1 validation cases (target recall {HEAD_RECALL:.0%} there)"}
(HD / "feature_definitions.json").write_text(json.dumps(FEATURE_DEFS, indent=1))
for name, (clf, thr) in HEAD.items():
    joblib.dump(clf, HD / f"warning_{name}.joblib")              # sklearn Pipeline: StandardScaler + LogisticRegression
    sc, lr = clf.named_steps["standardscaler"], clf.named_steps["logisticregression"]
    (HD / f"warning_{name}.json").write_text(json.dumps({"genco_model": name, "threshold": thr, "scaler_mean": sc.mean_.tolist(), "scaler_scale": sc.scale_.tolist(),
                                                         "coef": lr.coef_.ravel().tolist(), "intercept": float(lr.intercept_[0]), "C": 0.05, "class_weight": "balanced",
                                                         "sklearn": sklearn.__version__, "score": "1 / (1 + exp(-(coef . ((x - scaler_mean) / scaler_scale) + intercept)))"}, indent=1))
    np.savez_compressed(HD / f"features_{name}.npz", **{f"{lab}__X": X for (m_, lab), (X, y, ids) in FEATS_OUT.items() if m_ == name},
                        **{f"{lab}__y": y for (m_, lab), (X, y, ids) in FEATS_OUT.items() if m_ == name},
                        **{f"{lab}__ids": np.array(ids) for (m_, lab), (X, y, ids) in FEATS_OUT.items() if m_ == name})
pd.DataFrame(SCORES).to_csv(HD / "per_case_scores.csv", index=False)
# self-check: the exported JSON reproduces the pipeline's scores
for name, (clf, thr) in HEAD.items():
    J = json.loads((HD / f"warning_{name}.json").read_text()); X = FEATS_OUT[(name, "F1")][0]
    z = ((X - np.array(J["scaler_mean"])) / np.array(J["scaler_scale"])) @ np.array(J["coef"]) + J["intercept"]
    assert np.allclose(1 / (1 + np.exp(-z)), clf.predict_proba(X)[:, 1], atol=1e-9)
print(f"exported the warning: {len(HEAD)} fitted pipelines (joblib and plain JSON), thresholds, feature definitions, {len(SCORES)} per-case scores, feature matrices")

## 8. Figure, summary and output files

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
ax = axes[0]
fams = sorted(verified.family.unique()); x = np.arange(len(fams))
ax.bar(x - 0.2, [verified[verified.family == f].released_missed.mean() for f in fams], 0.4, label="released")
ax.bar(x + 0.2, [verified[verified.family == f].fine_tuned_missed.mean() for f in fams], 0.4, label="fine-tuned")
ax.set_xticks(x); ax.set_xticklabels(fams, rotation=30, fontsize=7); ax.set_ylabel("decisions missed per verified case"); ax.set_title("GENCO alone", fontsize=9); ax.legend(fontsize=7)
ax = axes[1]
lv = sorted(FG.level.unique()); x = np.arange(len(lv))
ax.bar(x - 0.2, [FG[FG.level == l].released_worst_err.median() for l in lv], 0.4, label="released")
ax.bar(x + 0.2, [FG[FG.level == l].fine_tuned_worst_err.median() for l in lv], 0.4, label="fine-tuned")
ax.set_xticks(x); ax.set_xticklabels([f"N-{l}" for l in lv]); ax.set_ylabel("worst-bus |Vm error|, median (p.u.)"); ax.set_title("released scenarios (forgetting)", fontsize=9); ax.legend(fontsize=7)
ax = axes[2]
sub = KS[KS.has_nose & ~KS.solvable_at_1 & KS.lam_rel.notna()]
for k, st in [("true", "k-"), ("released", "C0--"), ("fine_tuned", "C1-")]:
    gm = sub.groupby(sub.lam_rel.round(3))[f"{k}_min_vm"].median()
    ax.plot(gm.index, gm.values, st, label={"true": "AC solution"}.get(k, k.replace("_", "-")))
ax.axvline(0, color="grey", lw=0.8); ax.set_xlabel("load factor relative to the fold"); ax.set_ylabel("lowest voltage (median, p.u.)"); ax.set_title("unresolved storms near the fold", fontsize=9); ax.legend(fontsize=7)
ax = axes[3]
from sklearn.metrics import roc_curve
for name in MODELS:
    clf, thr = HEAD[name]; df = EV[EV.set == "F1"]
    X = np.stack([FEAT[("F1", int(s), name)] for s in df.sid]); y = (df.ref_status == "unresolved").to_numpy().astype(int)
    fpr, tpr, _ = roc_curve(y, clf.predict_proba(X)[:, 1]); ax.plot(fpr, tpr, label=f"head on {name.replace('_', '-')} GENCO")
ax.plot([0, 1], [0, 1], ":", color="grey"); ax.set_xlabel("false alarm rate"); ax.set_ylabel("recall of unresolved cases"); ax.set_title("warning (unresolved) on F1", fontsize=9); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(Path(OUT_DIR) / "fig_finetuned.png", dpi=150); plt.show()

In [ ]:
import platform, shutil
summary = {"frozen_protocol": T2_PROTOCOL, "checkpoint_sha256": CK_SHA, "endpoints_cases": E12.to_dict(orient="records"), "endpoint_forgetting": E3.to_dict(orient="records"),
           "endpoint_near_fold": E4, "endpoint_head": E5.to_dict(orient="records"), "secondary": SEC,
           "resolved_only_with_finetuned": (PCHK.to_dict(orient="records") if isinstance(PCHK, pd.DataFrame) else []),
           "graphkit": {"checked_out_commit": subprocess.run(["git", "-C", GRAPHKIT_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()},
           "versions": {"python": platform.python_version(), "numpy": np.__version__, "torch": torch.__version__, "device": str(DEVICE), "lightsim2grid": HAVE_KLU}}
json.dump(summary, open(Path(OUT_DIR) / "summary.json", "w"), indent=1, default=float)
print("VERDICT (frozen endpoints; every number with its interval in the files)")
for r_ in E12[E12.metric.isin(["missed", "worst_err"])].itertuples(index=False):
    print(f"- {r_.group}, {r_.metric}: released {r_.released_mean:.4f} -> fine-tuned {r_.fine_tuned_mean:.4f} per case (difference {r_.diff_mean:+.4f}, 95 % CI {r_.diff_ci_lo:+.4f} to {r_.diff_ci_hi:+.4f}; {r_.cases} cases)")
for r_ in E3.itertuples(index=False):
    print(f"- forgetting {r_.level}: median worst-bus error {r_.released_median_err:.4f} -> {r_.fine_tuned_median_err:.4f} p.u.; mean increase {r_.diff_mean_err:+.4f} (CI to {r_.diff_ci_hi:+.4f}); non-inferior: {r_.non_inferior}")
u = E4["unresolved_at_actual_load"]
print(f"- near the fold (unresolved storms): lowest voltage at the fold AC {u['lowest_v_at_fold']['true']:.3f} / released {u['lowest_v_at_fold']['released']:.3f} / fine-tuned {u['lowest_v_at_fold']['fine_tuned']:.3f} p.u.; "
      f"drop share released {u['drop_share_of_ac']['released']:.2f} -> fine-tuned {u['drop_share_of_ac']['fine_tuned']:.2f}; Newton from the fine-tuned state beyond the fold converged at {u['fine_tuned_newton_converged_beyond']} of {u['beyond_fold_points']} points")
for r_ in E5[E5.test == "F1"].itertuples(index=False):
    print(f"- head on {r_.model} GENCO, F1: AUROC {r_.auroc:.3f} (residual alone {r_.auroc_genco_residual_alone:.3f}, predicted violations alone {r_.auroc_genco_predicted_violations_alone:.3f}); recall {r_.recall_at_threshold:.2f} at a false-alarm rate of {r_.false_alarm_rate:.3f}")
print(f"- cases resolved only from the fine-tuned state: {SEC['fine_tuned']['resolved_only_with_this_genco']}")
shutil.make_archive(OUT_DIR, "zip", OUT_DIR)
print(f"written {OUT_DIR}.zip")
save_to_drive(OUT_DIR + ".zip")